# Drone Surveillance Intelligence Agent — Complete Technical Walkthrough


------------------------------------------------------------------------

## 1. Table of Contents

1.  [Executive Summary](#1-executive-summary)
2.  [Project Evolution & Planning](#2-project-evolution--planning)
3.  [Complete System Architecture](#3-complete-system-architecture)
4.  [Complete Folder & File
    Breakdown](#4-complete-folder--file-breakdown)
5.  [End-to-End Execution Flow](#5-end-to-end-execution-flow)
6.  [Full Data Flow Analysis](#6-full-data-flow-analysis)
7.  [Storage & State Management](#7-storage--state-management)
8.  [API, Service & Pipeline
    Breakdown](#8-api-service--pipeline-breakdown)
9.  [Component Interaction Matrix](#9-component-interaction-matrix)
10. [VLM Pipeline & frames.json Deep
    Dive](#10-vlm-pipeline--framesjson-deep-dive)
11. [Async Operations, Loops & Event
    Systems](#11-async-operations-loops--event-systems)
12. [Error Handling, Edge Cases &
    Pitfalls](#12-error-handling-edge-cases--pitfalls)
13. [Engineering Tradeoffs & Design
    Decisions](#13-engineering-tradeoffs--design-decisions)
14. [Dependency & Relationship
    Mapping](#14-dependency--relationship-mapping)
15. [Developer Onboarding Guide](#15-developer-onboarding-guide)
16. [Future Improvements](#16-future-improvements)
17. [Glossary](#17-glossary)

------------------------------------------------------------------------

## 2. Executive Summary

### 2.1 What This Project Is

The Drone Surveillance Intelligence Agent is a prototype AI surveillance
system that processes simulated drone telemetry and camera feeds to
detect, classify, and escalate security threats on a monitored property.
It is designed as an **AI engineering demonstration** — not a production
drone system — proving architectural sophistication, intelligent AI
integration, and clean system design without overengineering.

### 2.2 Core Capabilities

-   Ingests drone camera frame descriptions and telemetry (location,
    altitude, drone ID, timestamp)
-   Semantically classifies activities and entity types using embedding
    similarity — no keyword brittle matching
-   Builds **behavioral chains** that track recurring threats across
    time (a hooded figure at 22:00, same figure 12 hours later = same
    chain, not a new incident)
-   **Dormant chain reactivation**: prior session threats automatically
    resurface when similar observations appear
-   Applies a **risk gate** to skip expensive LLM calls on benign frames
-   Calls a free-tier Groq LLM (Llama 3.1 8B) only for high-risk frames
    that require threat reasoning
-   Generates structured security alerts with recommended actions
-   Exposes a real-time **Server-Sent Events (SSE) dashboard** and a
    **parallel chat interface** for property owner queries
-   Includes a **VLM subsystem** that can process real video files and
    generate `frames.json` using a lightweight vision-language model

### 2.3 Key Technologies

| Layer              | Technology                          | Why                                     |
|------------------------|------------------------|------------------------|
| Language           | Python 3.11+                        | AsyncIO support, ecosystem              |
| Web Framework      | FastAPI + Uvicorn                   | Async-native, SSE support               |
| Vector Database    | ChromaDB (persistent)               | Embedded, no server, metadata filtering |
| Embeddings         | fastembed (`BAAI/bge-small-en`)     | \~50MB, no torch, pure pip              |
| Graph Memory       | NetworkX + pickle                   | Zero-config, in-process, persists       |
| Structured Storage | SQLite (sqlite3)                    | Built-in Python, zero-config            |
| LLM Reasoning      | Groq API (llama-3.1-8b-instant)     | Free tier, \~200ms, structured output   |
| LLM Orchestration  | LangChain (Groq adapter)            | Prompt templates + output parsing only  |
| VLM Subsystem      | HuggingFaceTB/SmolVLM-256M-Instruct | Tiny 256M model, real frame captioning  |
| Async Queue        | Python asyncio.Queue                | Event bus, no extra dependencies        |
| UI                 | Vanilla HTML/CSS/JS                 | Zero build step, single file            |

### 2.4 System Design Philosophy

**Intelligence at the right layer.** The system uses LLMs where they
create genuine signal (threat reasoning, chat answers) and rule-free
semantic embeddings for classification. It explicitly avoids using LLMs
for tasks that deterministic code handles better.

**Minimal dependencies.** Seven core dependencies. No torch, no
transformers in the main pipeline, no LangChain memory modules, no
FAISS. Install time under three minutes.

**Persistent memory across sessions.** All state survives restarts.
Chains from yesterday are reactivation candidates today.

**Parallel, non-blocking UI.** The chat endpoint and the pipeline loop
run as independent async tasks. An operator query never blocks frame
processing.

------------------------------------------------------------------------

## 3. Project Evolution & Planning

### 3.1 The Assignment Context

The project originated as an AI engineering assignment requiring a
“Drone Security Analyst Agent.” The evaluation criteria were not about
building production drone infrastructure but about demonstrating:

-   Ability to architect AI systems with clear reasoning
-   Multimodal pipeline design (text descriptions as frame proxies)
-   Retrieval system design (structured + semantic + graph)
-   Intelligent LLM usage (not using AI for everything)
-   Clean documentation and engineering judgment

The assignment explicitly discouraged real drone control, live RTSP
streams, YOLO training, Kubernetes, or distributed systems — recognizing
that engineering clarity beats engineering scale at prototype stage.

### 3.2 Planning Phase 1 — Initial Concept

The initial concept proposed a 10-step linear pipeline:

    Input Ingestion → Observation Extraction → Multi-Layer Retrieval →
    Situation Context → Event Synthesis → Chain Management →
    Graph Update → Context Refinement → LLM Reasoning → Alert Engine

The first proposed stack was: FAISS + NetworkX + LangChain + SQLite.

The architecture started from a simple idea: a surveillance frame enters
the system, an event is generated, and the process ends. Initially, the
system treated every observation as an isolated incident.

The key shift happened when you realized real-world surveillance
intelligence is not linear. A suspicious observation may seem
unimportant at first but become meaningful later when new evidence
appears. For example, a person near a gate at 2 AM may not matter
initially, but if the same individual appears again near another
restricted area hours later, the earlier incident gains significance.

This led to the frame → event → chain reasoning model.

A frame is raw observational evidence. An event is the semantic
interpretation of that observation. A chain is an evolving behavioral
narrative connecting related events over time. From there, the
architecture naturally evolved into a multi-layer memory model:

-   Frame memory stores raw observational evidence.
-   Event memory stores semantic interpretations.
-   Chain memory stores evolving behavioral narratives.
-   Graph memory stores relational contextual knowledge.

**What was sound from the start:** - Separating LLM reasoning from
memory management (LLM is stateless, memory is external) - Multi-layer
retrieval (vector + graph + SQL) for different query types - The
behavioral chain concept — tracking recurring actors across time, not
just individual events

### 3.3 Planning Phase 2 — Stack Revision

Claude’s review of the initial plan identified three problems:

**LangChain overhead.** For a deterministic single-agent linear
pipeline, LangChain adds \~500ms import time and three abstraction
layers with zero benefit. The Anthropic SDK (later Groq SDK) does the
same job in ten lines.

**FAISS vs ChromaDB.** FAISS is file-based and requires custom
persistence glue. ChromaDB is a drop-in that provides persistence,
metadata filtering, and a cleaner query API out of the box.

**Missing pieces.** No configuration file (all thresholds hardcoded), no
simulation runner, no logging subsystem, no pinned requirements.

The revised lightweight stack:

| Component         | Before      | After                            | Reason                                    |
|------------------|------------------|------------------|------------------|
| Vector DB         | FAISS       | ChromaDB                         | Persistence + metadata filtering built-in |
| Graph             | NetworkX    | NetworkX (kept)                  | Already lightweight, saves as .gpickle    |
| Structured DB     | SQLite      | SQLite (kept)                    | Perfect fit, zero-config                  |
| Embeddings        | unspecified | fastembed BAAI/bge-small-en      | \~50MB, no torch                          |
| LLM               | unspecified | Groq llama-3.1-8b-instant        | Free tier, fast                           |
| LLM Orchestration | LangChain   | Direct SDK → then LangChain kept | User preference for prompt templates      |

LangChain was retained but scoped to prompt templating and output
parsing only — not for memory or retrieval (its most heavyweight
features).

### 3.4 Planning Phase 3 — Architectural Upgrade

After stack selection, three deeper architectural questions were raised:

**Question 1: Sequential vs. event-driven.** The original pipeline
processed every frame through all 10 steps in sequence. This means a
benign “empty parking lot” frame triggered the same LLM call as a
“masked intruder at night.” The answer was a **risk gate**: only frames
whose rule-based risk score exceeds 0.45 call the LLM. Cheap frames are
fast. Expensive frames escalate.

**Question 2: Parallel workers.** Rather than sequential steps,
retrieval, synthesis, and chain state fetching can run concurrently
since they don’t depend on each other. Three async tasks fire
simultaneously and the context assembler waits for all three before
proceeding.

**Question 3: Dormant chain reactivation.** The “same person yesterday
and today” problem. A suspicious entity detected at 22:00, inactive for
12 hours, reappears at 10:00 the next day. The system should wake up the
prior chain, not create a new one. This required: - Chains are marked
`dormant` rather than deleted when the pipeline ends - On startup,
active chains older than one hour become dormant automatically - New
observations query dormant chains by vector similarity + entity type +
location adjacency - A match above the chain link threshold (`0.60`)
reactivates the dormant chain and resumes escalation from prior state

### 3.5 Planning Phase 4 — Robustness Fixes

After the first real run with 12 frames, four issues emerged that
informed the final implementation:

**Issue: Entity extraction fragility.** String matching meant “Hooded
man” and “hooded figure” were treated as different entities, preventing
chain linking. The fix was to replace all string-based entity extraction
with **semantic embedding classification** against a fixed set of
natural language categories. Any description of a suspicious hooded
person maps to `"person behaving suspiciously near perimeter"`
regardless of the exact words used.

**Issue: Chain matching fragility.** Chain linking used substring
overlap. Two observations about the same entity with different words
didn’t link. The fix was to compare **embedding similarity** between the
current observation and the chain’s stored embedding, using cosine
distance as the primary signal.

**Issue: Risk de-escalation on repeated presence.** A circling truck
(event risk 0.2) on a chain already at 0.495 caused risk to *drop*
because of the weighted average. Repeated presence should escalate, not
de-escalate. The fix introduced a **repeat pressure bonus**: each
additional event in a chain adds a small escalation factor, and
de-escalation is floored at 95% of prior chain risk.

**Issue: Security guard chains polluting alert logic.** Security guard
patrols and authorized delivery vehicles were building up risk scores
and occasionally triggering LLM calls. The fix introduced a
**disposition system**: chains are classified as `benign`, `neutral`, or
`suspicious`. Benign chains have a hard risk cap (0.32) — the LLM is
never called, and no alerts are generated.

### 3.6 Planning Phase 5 — UI Addition

The final planning phase added a real-time dashboard and chat interface.
The key insight was that the UI and the pipeline are completely
decoupled:

-   Pipeline runs as a background `asyncio.Task`, pushing events to a
    shared queue
-   The SSE endpoint drains that queue and streams to the browser
-   The chat endpoint reads from memory (SQLite + graph) independently,
    calls the LLM, and returns an answer — with zero interaction with
    the pipeline loop

This architecture means the operator can send queries mid-pipeline run
without any locking or coordination.

------------------------------------------------------------------------

## 4. Complete System Architecture

### 4.1 High-Level Architecture Overview

    ┌──────────────────────────────────────────────────────────────────┐
    │                    ENTRY POINTS                                  │
    │  ┌─────────────────────┐      ┌──────────────────────────────┐  │
    │  │  CLI: python -m     │      │  Web: uvicorn app.api:app    │  │
    │  │  app.pipeline       │      │  port 8000                   │  │
    │  └──────────┬──────────┘      └──────────────┬───────────────┘  │
    └─────────────┼──────────────────────────────── ┼ ────────────────┘
                  │                                 │
                  ▼                                 ▼
    ┌─────────────────────────────────────────────────────────────────┐
    │                    PIPELINE ORCHESTRATOR                         │
    │                    app/pipeline.py                               │
    │   init_hub() → load frames → for each frame: process_frame()    │
    └───────────────────────┬─────────────────────────────────────────┘
                            │
              ┌─────────────┴──────────────┐
              │   INGESTION LAYER          │
              │   ingestion/               │
              │   frame_input.py           │
              │   telemetry_input.py       │
              │   observation_extractor.py │
              └─────────────┬──────────────┘
                            │  observation dict (with embedding)
                            ▼
    ┌───────────────────────────────────────────────────────────────────┐
    │              PARALLEL WORKERS (asyncio.gather)                    │
    │  ┌──────────────────┐ ┌──────────────────┐ ┌──────────────────┐  │
    │  │ retrieval_worker │ │ synthesis_worker  │ │  chain_worker    │  │
    │  │                  │ │                  │ │                  │  │
    │  │ Frame retriever  │ │ Event synthesizer│ │ SQL chain fetch  │  │
    │  │ Event retriever  │ │ Risk scoring     │ │ Dormant check    │  │
    │  │ Chain retriever  │ │                  │ │                  │  │
    │  │ Graph retriever  │ │                  │ │                  │  │
    │  └──────────────────┘ └──────────────────┘ └──────────────────┘  │
    └───────────────────────────────┬───────────────────────────────────┘
                                    │  3 result dicts
                                    ▼
    ┌───────────────────────────────────────────────────────────────────┐
    │              CONTEXT ASSEMBLER                                    │
    │              engine/context_assembler.py                          │
    │                                                                   │
    │  Merge candidates → Score chain matches → Link/Reactivate/Create  │
    │  Apply disposition → Apply risk cap → Persist to all 3 stores    │
    └───────────────────────────────┬───────────────────────────────────┘
                                    │  situation dict
                                    ▼
                        ┌───────────────────────┐
                        │    RISK GATE          │
                        │  engine/risk_gate.py  │
                        │  risk >= 0.45?        │
                        └───────┬───────┬───────┘
                            NO  │       │  YES
                                │       ▼
                                │  ┌─────────────────────┐
                                │  │   LLM REASONER      │
                                │  │ reasoning/llm_reasoner│
                                │  │ Groq API call        │
                                │  └──────────┬───────────┘
                                │             │
                                └────┬────────┘
                                     │  llm_output dict
                                     ▼
                        ┌───────────────────────────┐
                        │      ALERT ENGINE         │
                        │   alerts/alert_engine.py  │
                        │   risk >= 0.70 → alert    │
                        └───────────────────────────┘

### 4.2 Memory Architecture

Three orthogonal storage systems, each optimized for different query
types:

    ┌─────────────────────────────────────────────────────────────────────┐
    │                         MEMORY HUB                                  │
    │                      memory/memory_hub.py                           │
    │                                                                     │
    │  ┌───────────────┐   ┌──────────────────┐   ┌───────────────────┐  │
    │  │    SQLite     │   │    ChromaDB      │   │    NetworkX       │  │
    │  │               │   │                  │   │    Graph          │  │
    │  │  Ground truth │   │  Semantic lookup │   │  Relationship     │  │
    │  │  frames table │   │  frames_coll     │   │  traversal        │  │
    │  │  events table │   │  events_coll     │   │                   │  │
    │  │  chains table │   │  chains_coll     │   │  Nodes:           │  │
    │  │  alerts table │   │                  │   │  entities,events, │  │
    │  │               │   │  "What looked    │   │  chains,alerts,   │  │
    │  │  "How many    │   │  like this?"     │   │  locations        │  │
    │  │  events in    │   │                  │   │                   │  │
    │  │  chain 3?"    │   │                  │   │  "Is this entity  │  │
    │  │               │   │                  │   │  linked to a      │  │
    │  │  Exact +      │   │  Approximate     │   │  prior alert?"    │  │
    │  │  range query  │   │  cosine search   │   │                   │  │
    │  └───────────────┘   └──────────────────┘   └───────────────────┘  │
    │  data/db/             data/chroma/           data/graph/            │
    │  surveillance.db      (collection dirs)      memory.gpickle         │
    └─────────────────────────────────────────────────────────────────────┘

### 4.3 Layered Architecture Diagram

    Layer 1 — Entry         CLI / FastAPI Web Server
                                   │
    Layer 2 — Ingestion     frame_input → telemetry_input → observation_extractor
                                   │
    Layer 3 — Concurrency   asyncio.gather (retrieval / synthesis / chain)
                                   │
    Layer 4 — Assembly      context_assembler (merge, score, persist)
                                   │
    Layer 5 — Gate          risk_gate (skip LLM if risk < 0.45)
                                   │
    Layer 6 — Reasoning     llm_reasoner (Groq API → structured JSON)
                                   │
    Layer 7 — Action        alert_engine (threshold check, persist, log)
                                   │
    Layer 8 — UI            FastAPI SSE /stream + /chat + / (HTML dashboard)

### 4.4 Data Store Relationship

    frames ──────────────────── events ──────────── chains ──────── alerts
      │                            │                    │
      │ frame_id FK                │ chain_id FK         │ chain_id FK
      │                            │                    │ frame_id FK
      └──── ChromaDB               └── ChromaDB          └── ChromaDB (events_coll)
           frames_collection            events_collection      chains_collection
           
    entities / locations / chains / alerts ──── NetworkX DiGraph nodes
    relationships (involved_in, part_of,
    escalated_to, observed_at) ──────────────── NetworkX DiGraph edges

------------------------------------------------------------------------

## 5. Complete Folder & File Breakdown

### 5.1 Directory Tree

    srvlnc_agent_clean/
    │
    ├── app/                          # Main application package
    │   ├── alerts/
    │   │   ├── __init__.py
    │   │   └── alert_engine.py       # Threshold check → generate alerts
    │   │
    │   ├── engine/
    │   │   ├── __init__.py
    │   │   ├── context_assembler.py  # Core intelligence: chain resolution
    │   │   ├── event_synthesizer.py  # Semantic risk scoring per frame
    │   │   └── risk_gate.py          # LLM call gate function
    │   │
    │   ├── ingestion/
    │   │   ├── __init__.py
    │   │   ├── frame_input.py        # Load frames.json
    │   │   ├── telemetry_input.py    # Load telemetry.json → indexed by frame_id
    │   │   └── observation_extractor.py  # Semantic classification + embedding
    │   │
    │   ├── memory/
    │   │   ├── __init__.py
    │   │   ├── graph_store.py        # NetworkX DiGraph wrapper
    │   │   ├── memory_hub.py         # Unified access to all 3 stores
    │   │   ├── sqlite_store.py       # SQLite CRUD for all 4 tables
    │   │   └── vector_store.py       # ChromaDB 3-collection wrapper
    │   │
    │   ├── reasoning/
    │   │   ├── __init__.py
    │   │   └── llm_reasoner.py       # LangChain-Groq prompt + parse
    │   │
    │   ├── retrieval/
    │   │   ├── __init__.py
    │   │   ├── chain_retriever.py    # ChromaDB chains_collection query
    │   │   ├── event_retriever.py    # ChromaDB events_collection query
    │   │   ├── frame_retriever.py    # ChromaDB frames_collection query
    │   │   ├── graph_retriever.py    # NetworkX ego-graph query
    │   │   └── retriever_base.py     # Shared similarity floor filter
    │   │
    │   ├── utils/
    │   │   ├── __init__.py
    │   │   ├── embeddings.py         # fastembed singleton wrapper
    │   │   └── logger.py             # Structured logging factory
    │   │
    │   ├── workers/
    │   │   ├── __init__.py
    │   │   ├── chain_worker.py       # Async: SQL chain state snapshot
    │   │   ├── retrieval_worker.py   # Async: 4 retriever calls
    │   │   └── synthesis_worker.py   # Async: event synthesis
    │   │
    │   ├── api.py                    # FastAPI app: /stream /chat /
    │   ├── bus.py                    # asyncio.Queue pair (observation_queue, result_queue)
    │   └── pipeline.py               # Orchestrator: init_hub(), process_frame(), run()
    │
    ├── data/
    │   ├── db/
    │   │   └── surveillance.db       # SQLite database (generated at runtime)
    │   ├── graph/
    │   │   └── memory.gpickle        # Pickled NetworkX DiGraph (generated)
    │   ├── chroma/                   # ChromaDB persistent collections (generated)
    │   └── simulation/
    │       ├── frames.json           # 30 simulated drone frame descriptions
    │       └── telemetry.json        # 30 telemetry records (time/location/altitude)
    │
    ├── ui/
    │   └── index.html                # Single-file dashboard (dark theme, SSE client)
    │
    ├── vlm/
    │   ├── main.py                   # FastAPI VLM server (video → frames.json)
    │   ├── uploads/                  # Uploaded video files (temp)
    │   ├── outputs/                  # Generated frames.json files
    │   └── temp_frames/              # Extracted frame images (temp)
    │
    ├── config.py                     # All thresholds, paths, model names
    ├── requirements.txt              # Pinned dependencies
    └── .env                          # GROQ_API_KEY (not committed)

### 5.2 File-by-File Analysis

#### `config.py`

**Purpose:** Single source of truth for all tunable parameters. Nothing
in the pipeline hardcodes a threshold.

**Why it exists:** In the early planning phase, risk thresholds were
scattered in logic files. A change to `LLM_RISK_THRESHOLD` required
hunting through multiple files. Centralizing them means calibrating the
system is a single-file edit.

**Key contents:**

``` python
# Risk thresholds — the three gates in the pipeline
LLM_RISK_THRESHOLD  = 0.45   # Below this: skip LLM entirely
ALERT_THRESHOLD     = 0.70   # Below this: no formal alert generated
CHAIN_LINK_THRESHOLD = 0.60  # Below this: create new chain, don't link

# Chain memory window
CHAIN_REACTIVATION_WINDOW = 86400  # 24 hours — dormant chains within window are candidates

# Restricted locations — elevate risk score
RESTRICTED_LOCATIONS = ["North Perimeter", "Garage Sector", "Server Room", "Gate Alpha"]

# Location adjacency — for chain matching ("near" counts as "at")
LOCATION_ADJACENCY = {
    "North Perimeter": ["Gate Alpha", "East Fence"],
    "Garage Sector":   ["Loading Bay", "South Exit"],
}

# Night hours (minutes from 00:00)
NIGHT_START = 1200   # 20:00
NIGHT_END   = 360    # 06:00

# Entity embedding similarity threshold for chain matching
ENTITY_MATCH_THRESHOLD = 0.72
```

**Consumers:** Nearly every module in the system imports `config`
directly.

------------------------------------------------------------------------

#### `app/ingestion/frame_input.py`

**Purpose:** Loads `frames.json` from disk and returns it as a Python
list.

**Input:** File path (defaults `data/simulation/frames.json`)  
**Output:** `list[dict]` — each dict has `frame_id: int` and
`description: str`

**Why it’s separate:** The ingestion layer is intentionally thin.
`frame_input.py` only loads data; it does no transformation. This makes
it trivial to swap the data source (database, Kafka topic, API endpoint)
without touching any other component.

------------------------------------------------------------------------

#### `app/ingestion/telemetry_input.py`

**Purpose:** Loads `telemetry.json` and returns it as a dictionary
indexed by `frame_id`.

**Input:** File path  
**Output:** `dict[int, dict]` — key is `frame_id`, value has `time`,
`location`, `altitude`, `drone_id`

**Key design:** Building the index at load time
(`{e["frame_id"]: e for e in entries}`) means each per-frame telemetry
lookup is O(1) instead of O(n) per frame. On 30 frames this is trivial;
on 10,000 frames it matters.

------------------------------------------------------------------------

#### `app/ingestion/observation_extractor.py`

**Purpose:** The most important ingestion file. Converts a raw frame
description + telemetry dict into a rich structured observation that
every downstream component uses.

**Inputs:** - `frame: dict` — `{frame_id, description}` -
`telemetry: dict` — `{time, location, altitude, drone_id}`

**Output:** An `observation dict` with 14 fields:

``` python
{
    "frame_id":              14,
    "raw_description":       "Hooded man crouching near perimeter fence line",
    "activity":              "crouching or hiding near fence or perimeter",
    "activity_conf":         0.82,
    "entity_type":           "person behaving suspiciously near perimeter",
    "entity_conf":           0.79,
    "entity_fingerprint":    "Hooded man",            # specific identity fragment
    "fingerprint_embedding": [...],                  # 384-dim vector of fingerprint
    "entities":              ["person behaving suspiciously near perimeter"],
    "location":              "North Perimeter",
    "time_str":              "22:47",
    "time_minutes":          1367,
    "altitude":              8,
    "drone_id":              "D-02",
    "embedding":             [...],                  # 384-dim vector of full text
}
```

**Semantic Classification (the key architectural choice):**

Instead of keyword matching, the extractor uses **embedding similarity
against fixed category lists**:

    ACTIVITY_CATEGORIES = [
        "crouching or hiding near fence or perimeter",
        "walking or patrolling normally",
        "vehicle loitering or circling",
        ...12 total categories
    ]

    ENTITY_CATEGORIES = [
        "person behaving suspiciously near perimeter",
        "authorized security personnel on patrol",
        "vehicle parked or stationary",
        ...9 total categories
    ]

For any input description, the extractor: 1. Embeds the description text
2. Computes cosine similarity against all category embeddings (cached
after first call) 3. Returns the highest-scoring category

This means `"hooded man crouching"`, `"hooded figure moving"`,
`"masked individual surveilling"` all classify to the same semantic
bucket — enabling reliable chain linking.

**Entity Fingerprinting:**

Beyond the broad entity type, the extractor generates a
`entity_fingerprint` — a short, specific phrase extracted from the
description (e.g., `"Hooded man"`, `"Blue pickup truck"`). This
fingerprint is separately embedded and stored in chains, enabling
fine-grained identity matching: a blue pickup and a silver sedan share
the same broad entity type (`"vehicle parked or stationary"`) but have
very different fingerprint embeddings, preventing them from merging into
the same chain.

------------------------------------------------------------------------

#### `app/utils/embeddings.py`

**Purpose:** Singleton wrapper around fastembed’s `TextEmbedding` model.

**Why singleton:** The fastembed model is \~50MB and takes several
seconds to load. Loading it once at first use and caching globally
prevents per-call re-initialization.

``` python
_model = None
def get_model() -> TextEmbedding:
    global _model
    if _model is None:
        _model = TextEmbedding(model_name=config.EMBED_MODEL)
    return _model

def embed(text: str) -> list:
    model = get_model()
    result = list(model.embed([text]))
    return result[0].tolist()
```

**Output shape:** 384-dimensional float list (BAAI/bge-small-en output
dimension).

**Consumers:** `observation_extractor.py`, `event_synthesizer.py`,
`context_assembler.py`, all retrievers.

------------------------------------------------------------------------

#### `app/utils/logger.py`

**Purpose:** Factory for named Python loggers with consistent
formatting.

**Output format:**
`[2026-05-16 22:31:45] [DEBUG] [context_assembler] Situation assembled...`

**`log_json` helper:** Accepts a logger, a label string, and a dict —
serializes the dict to indented JSON and logs at DEBUG level. Used in
`alert_engine.py` to dump full alert payloads.

------------------------------------------------------------------------

#### `app/memory/memory_hub.py`

**Purpose:** Unified access point for all three storage systems.

**Why it exists:** Every worker, assembler, and engine receives a single
`memory_hub` object rather than three separate store objects. This has
two benefits: 1. Dependency injection is one argument, not three 2.
Tests can mock the entire storage layer with a single object

``` python
class MemoryHub:
    def __init__(self, db_path, chroma_path, graph_path):
        os.makedirs(os.path.dirname(db_path), exist_ok=True)
        os.makedirs(chroma_path, exist_ok=True)
        os.makedirs(os.path.dirname(graph_path), exist_ok=True)
        self.sql    = SQLiteStore(db_path)
        self.vector = VectorStore(chroma_path)
        self.graph  = GraphStore(graph_path)
```

**Access pattern:** `memory_hub.sql.insert_event(event)`,
`memory_hub.vector.query_frames(embedding)`,
`memory_hub.graph.add_edges(triples)`.

------------------------------------------------------------------------

#### `app/memory/sqlite_store.py`

**Purpose:** Complete SQLite CRUD layer for all four tables.

**Schema:**

``` sql
CREATE TABLE frames (
    frame_id     INTEGER PRIMARY KEY,
    drone_id     TEXT,
    description  TEXT,
    location     TEXT,
    time_str     TEXT,
    time_minutes INTEGER,
    altitude     REAL,
    activity     TEXT,
    risk_score   REAL,
    created_at   TEXT
);

CREATE TABLE events (
    event_id     INTEGER PRIMARY KEY AUTOINCREMENT,
    frame_id     INTEGER,
    chain_id     INTEGER,
    type         TEXT,
    entities     TEXT,   -- JSON array as string
    location     TEXT,
    time_str     TEXT,
    time_minutes INTEGER,
    risk_score   REAL,
    llm_risk     REAL,   -- LLM override (NULL if LLM not called)
    created_at   TEXT
);

CREATE TABLE chains (
    chain_id     INTEGER PRIMARY KEY AUTOINCREMENT,
    status       TEXT,   -- active / dormant / escalated
    narrative    TEXT,
    entities     TEXT,   -- JSON array
    locations    TEXT,   -- JSON array
    event_ids    TEXT,   -- JSON array
    risk_score   REAL,
    created_at   TEXT,
    last_updated TEXT
);

CREATE TABLE alerts (
    alert_id           INTEGER PRIMARY KEY AUTOINCREMENT,
    chain_id           INTEGER,
    frame_id           INTEGER,
    event_id           INTEGER,
    risk_level         REAL,
    suspiciousness     TEXT,
    reasoning          TEXT,   -- JSON array
    recommended_action TEXT,
    timestamp          TEXT
);
```

**Key methods and their roles:**

| Method                                      | Purpose                                                  |
|------------------------------------|------------------------------------|
| `insert_frame(obs)`                         | Stores raw observation — idempotent (INSERT OR REPLACE)  |
| `insert_event(event) → int`                 | Creates event row, returns `event_id`                    |
| `update_event_chain(event_id, chain_id)`    | Backfills `chain_id` after chain resolution              |
| `update_event_llm_risk(event_id, llm_risk)` | Records LLM override after reasoning                     |
| `insert_chain(chain) → int`                 | Creates chain, returns `chain_id`                        |
| `update_chain(chain)`                       | Upserts chain state changes                              |
| `get_events_for_chain(chain_id)`            | Full event history for a chain — used on reactivation    |
| `get_active_and_dormant_chains()`           | Returns latest 20 non-escalated chains                   |
| `get_dormant_chains_within_window(sec)`     | Dormant chains updated within the reactivation window    |
| `mark_old_chains_dormant(max_age_sec)`      | Transitions stale active chains to dormant at startup    |
| `get_processed_frame_ids() → set`           | Returns frame IDs already in DB — prevents re-processing |
| `get_recent_alerts(limit)`                  | Latest N alerts for chat context                         |

**Connection settings:** `check_same_thread=False` allows the same
SQLite connection to be used from multiple async coroutines.
`row_factory = sqlite3.Row` makes row access by column name instead of
index. JSON arrays (entities, locations, event_ids) are serialized with
`json.dumps()` on write and `json.loads()` on read.

------------------------------------------------------------------------

#### `app/memory/vector_store.py`

**Purpose:** ChromaDB wrapper managing three named collections.

**Collections:**

| Collection          | Indexed data                                                 | Used for                                        |
|------------------------|------------------------|------------------------|
| `frames_collection` | Per-frame embedding of `activity + location + description`   | “What past frames look visually similar?”       |
| `events_collection` | Per-event embedding of `event_type + location + description` | “What past behavioral events resemble now?”     |
| `chains_collection` | Per-chain embedding of `narrative + entities + locations`    | “Are there dormant chains whose story matches?” |

**ChromaDB telemetry suppression** (applied at module top):

``` python
import os, logging
os.environ["ANONYMIZED_TELEMETRY"] = "false"
logging.getLogger("chromadb.telemetry").setLevel(logging.ERROR)
logging.getLogger("chromadb").setLevel(logging.WARNING)
```

Without this, ChromaDB version 0.5.3 emits a telemetry error on every
query due to an API signature mismatch in its internal capture function.

**Query semantics:** ChromaDB returns distances (not similarities). The
wrapper converts: `similarity = 1 - distance`. Results below
`config.VECTOR_SIMILARITY_FLOOR` (0.55) are filtered out by
`retriever_base.py`.

------------------------------------------------------------------------

#### `app/memory/graph_store.py`

**Purpose:** NetworkX directed graph wrapper with pickle persistence.

**Graph structure:**

    Nodes: entities (str), "event_N" (str), "chain_N" (str), "alert_N" (str), locations (str)
    Edges: typed relationships (relation attribute on each edge)

**Relationship types:**

| Relationship   | Source → Target         | Meaning                                |
|------------------------|------------------------|------------------------|
| `involved_in`  | entity → chain_N        | This entity participated in this chain |
| `part_of`      | event_N → chain_N       | This event belongs to this chain       |
| `observed_at`  | entity/chain → location | Spatial association                    |
| `escalated_to` | chain_N → alert_N       | Chain generated an alert               |
| `triggered_by` | alert_N → event_N       | Alert caused by specific event         |

**`get_ego_graph(entity, radius=2)`:** Returns all nodes within 2 hops
of an entity. This answers multi-hop questions: “Is this entity linked
to any chain that previously escalated to an alert?” in a single call.

**Persistence:** The graph is saved as a pickle file (`memory.gpickle`)
after every write (`graph_store.save()`). On startup, it’s loaded back
into memory. The entire graph lives in-process for the session — never
accessed remotely. For 30 frames this is \~50 nodes and \~100 edges —
trivially small.

------------------------------------------------------------------------

#### `app/engine/event_synthesizer.py`

**Purpose:** Converts a raw observation into a typed behavioral event
with a risk score.

**Inputs:** `observation dict`, `memory_hub`  
**Output:** `event dict` with `type`, `entities`, `risk_score`,
`embedding`

**Risk scoring architecture — three-layer system:**

**Layer 1: Semantic descriptor similarity**

The synthesizer compares the observation embedding against high-risk and
low-risk descriptor embeddings:

``` python
HIGH_RISK_DESCRIPTORS = [
    "masked individual crouching near restricted perimeter at night",
    "hooded person hiding near secure fence line",
    "person attempting perimeter breach",
    ...
]
LOW_RISK_DESCRIPTORS = [
    "authorized security patrol",
    "delivery vehicle unloading at loading bay",
    "empty area no suspicious activity",
    ...
]
```

The semantic delta (`high_max - low_max`) contributes a bonus of 0.15
(strong signal) or 0.08 (moderate signal).

**Layer 2: Keyword suspicion markers**

Specific suspicious descriptors add weighted bonuses:

``` python
suspicious_markers = {
    "hooded": 0.16, "masked": 0.18, "crouching": 0.20,
    "hiding": 0.20, "tampering": 0.22, "climbing": 0.24,
    "breach": 0.24, "surveilling": 0.18, ...
}
```

**Layer 3: Contextual overrides**

-   `clear_markers` (security sweep, patrol, all clear) → force base
    risk to 0.25
-   Vehicle circling/loitering → floor risk at 0.48
-   Vehicle parked/stationary → cap risk at 0.32

**Layer 4: Contextual bonuses** (only applied if base risk \> 0.32)

-   Night time: +0.08
-   Restricted location: +0.05
-   Known suspicious entity in graph: +0.08

**Why hybrid?** Pure semantic similarity occasionally misses specific
high-threat words. Pure keyword matching misses paraphrases. The hybrid
(semantic base + keyword markers + contextual bonuses) is more robust
than either alone.

**Security response suppressor:** Observations about security guards or
authorized personnel have all bonuses zeroed and base risk forced to
0.25. This prevents the security response to a threat from being treated
as a new threat.

------------------------------------------------------------------------

#### `app/engine/context_assembler.py`

**Purpose:** The brain of the system. Receives all three worker results,
resolves the chain decision, persists everything, and returns the
complete situation object.

**This is the most complex and important file in the codebase.**

**Chain matching — four-signal scoring:**

``` python
def _score_chain_match(chain, observation, vector_similarity) -> float:
    fp_sim = _fingerprint_similarity(chain, observation)
    fp_score      = 0.35 * fp_sim              # Specific entity identity
    entity_score  = 0.15 if broad_category_match else 0.0  # Person vs vehicle gate
    location_score = 0.20 if location_match else 0.0       # Same/adjacent zone
    vector_score  = 0.30 * vector_similarity    # ChromaDB semantic cosine
    return fp_score + entity_score + location_score + vector_score
```

**Why fingerprint similarity dominates (0.35 weight):**

Fingerprints are short specific phrases (“Hooded man”, “Blue pickup
truck”). Comparing their embeddings directly is more discriminating than
comparing full observation embeddings, which blend activity, location,
and description into a single vector. A blue pickup truck and a silver
sedan have similar full observation embeddings (both are vehicles at
similar locations) but very different fingerprint embeddings.

**Hard gates before scoring:**

1.  **Category mismatch gate:** If a chain tracks a vehicle and the new
    observation is about a person, the chain is skipped entirely.
    Persons and vehicles cannot share a chain.
2.  **Risk floor gate:** If the event’s risk score is ≤ 0.35 (genuinely
    benign — empty lot, morning arrival), the observation doesn’t link
    to any chain. It’s recorded but treated as background.
3.  **Security response isolation gate:** Security guard observations
    don’t link to suspicious perimeter chains. Security arriving at a
    scene creates its own chain, not a continuation of the intruder
    chain.

**Disposition system:**

Every chain carries a `disposition` field: `benign`, `neutral`, or
`suspicious`.

| Disposition  | Risk cap | LLM called?        | Alert possible?        |
|--------------|----------|--------------------|------------------------|
| `benign`     | 0.32     | Never              | Never                  |
| `neutral`    | 0.65     | Maybe (if \> 0.45) | No (threshold is 0.70) |
| `suspicious` | None     | Yes                | Yes                    |

Initial disposition is set when a chain is created: - Entity is
authorized (security, delivery, maintenance) AND risk ≤ 0.45 →
`benign` - Event risk ≥ 0.55 → `suspicious` - Otherwise → `neutral`

Disposition transitions: - `benign` → `neutral`: if a high event (risk
0.52-0.64) arrives - `benign` → `suspicious`: only if risk ≥ 0.65 (a
genuinely threatening event near an authorized entity) - `neutral` →
`suspicious`: if risk ≥ 0.55 - `suspicious` → never downgrades

**Risk recalculation (escalation-aware):**

``` python
repeat_pressure = min(0.07 * event_count, 0.35)  # More events = more pressure
if disposition == "suspicious":
    repeat_pressure += 0.10   # Extra escalation for confirmed threats

if event_risk >= chain_risk:
    new_risk = 0.40 * chain_risk + 0.60 * event_risk + repeat_pressure
else:
    # De-escalation is slow — floor at 97% of prior risk
    blended  = 0.65 * chain_risk + 0.35 * event_risk
    floored  = chain_risk * 0.97
    new_risk = max(blended, floored) + repeat_pressure
```

This ensures a circling truck never accidentally de-escalates a chain
that’s already been confirmed suspicious.

------------------------------------------------------------------------

#### `app/engine/risk_gate.py`

**Purpose:** Single function, single responsibility.

``` python
def should_reason(situation: dict) -> bool:
    risk = situation["chain"]["risk_score"]
    result = risk >= config.LLM_RISK_THRESHOLD
    logger.info(f"Risk gate: score={risk} → {'PASS' if result else 'SKIP'}")
    return result
```

**Why a gate?** In a real surveillance scenario, most frames are benign
(a parked authorized vehicle, an empty lot, a staff member arriving).
Calling the LLM for every frame would cost money, add latency, and
produce meaningless “this is fine” outputs. The gate ensures LLM calls
are reserved for genuinely ambiguous or elevated situations.

------------------------------------------------------------------------

#### `app/reasoning/llm_reasoner.py`

**Purpose:** Calls the Groq API via LangChain, formats the situation
into a structured prompt, and parses the LLM response as JSON.

**LLM choice: Groq llama-3.1-8b-instant**

-   Free tier with generous limits
-   \~200ms response time
-   8B parameter model — sufficient for structured reasoning
-   Downside: no guaranteed structured output (unlike Anthropic’s API) —
    parsing is defensive

**Prompt design:**

The prompt provides: 1. Current observation (description, location,
time, activity type, entities) 2. Behavioral chain context (narrative,
status, event count, risk score) 3. Historical context (similar past
frames, similar past events, graph relations)

Expected output JSON:

``` json
{
    "suspiciousness": "high",
    "risk_level": 0.88,
    "reasoning": ["Point 1", "Point 2", "Point 3"],
    "recommended_action": "Dispatch security to North Perimeter",
    "confidence": "high"
}
```

**Defensive parsing:**

``` python
def _parse_response(text: str) -> dict:
    text = text.strip()
    if text.startswith("```"):                    # Strip markdown code fences
        lines = text.split("\n")
        text = "\n".join(lines[1:-1]) if len(lines) > 2 else text
    return json.loads(text)
```

If parsing fails (LLM returns prose, API error, rate limit), the
fallback returns a medium-risk, low-confidence response using the
rule-based chain score, rather than crashing.

------------------------------------------------------------------------

#### `app/alerts/alert_engine.py`

**Purpose:** Final gate — checks LLM output against `ALERT_THRESHOLD`
(0.70), generates and persists the alert if crossed.

**Alert generation sequence:** 1. Update event’s `llm_risk` field in
SQLite (even if no alert generated) 2. If `risk_level < 0.70`: log and
return `None` 3. Build alert dict and insert into `alerts` table 4. Set
chain `status = "escalated"`, update risk 5. Add graph edges:
`chain → escalated_to → alert`, `alert → triggered_by → event` 6. Save
graph pickle 7. Log full alert JSON at DEBUG level

**Why separate LLM threshold (0.45) from alert threshold (0.70)?**

The LLM is called for “interesting” situations (risk ≥ 0.45). The LLM
may decide the situation is actually less dangerous than the rule-based
score suggested (confidence: low) and set `risk_level = 0.50`. This
doesn’t trigger an alert but the reasoning is recorded. Only when the
LLM confirms high suspicion (≥ 0.70) does a formal alert fire.

------------------------------------------------------------------------

#### `app/workers/retrieval_worker.py`

**Purpose:** Async task that runs all four retrievers in sequence and
returns a combined result dict.

The four retrievers run sequentially (not in parallel) because they’re
all fast local operations (ChromaDB in-process, NetworkX in-memory). The
parallelism is at the worker level (retrieval vs synthesis vs chain),
not within a worker.

------------------------------------------------------------------------

#### `app/workers/synthesis_worker.py`

**Purpose:** Async task that calls `event_synthesizer.synthesize()` and
returns the event dict.

Thin wrapper — exists to make synthesis independently schedulable as an
async task.

------------------------------------------------------------------------

#### `app/workers/chain_worker.py`

**Purpose:** Async task that fetches current chain state from SQLite.

Fetches both active/dormant chains (candidates for linking) and dormant
chains within the reactivation window (prioritized candidates). Returns
both sets. The actual chain decision happens in the assembler after all
workers complete.

**Why not wait for synthesis?** Worker C starts simultaneously with
Workers A and B. It doesn’t need the synthesized event — it only needs
to snapshot chain state. The assembler’s chain decision requires both
the event (from B) and the candidates (from A and C), so the assembler
waits. Workers don’t need to wait for each other.

------------------------------------------------------------------------

#### `app/pipeline.py`

**Purpose:** Top-level orchestrator for both CLI and API modes.

**`init_hub()`:** Creates data directories, initializes MemoryHub, marks
stale chains dormant. Shared between CLI `run()` and API `lifespan()` —
same initialization path regardless of entry point.

**`process_frame(observation, memory_hub)`:** The per-frame processing
function. Fires three async tasks, waits for all three, assembles the
situation, gates the LLM, runs the alert engine, returns the complete
result.

**`run()`:** CLI entry point. Loads frames + telemetry, iterates, calls
`process_frame`, prints formatted output.

------------------------------------------------------------------------

#### `app/api.py`

**Purpose:** FastAPI application with three endpoints. Also manages the
pipeline background task.

**Lifespan manager:**

``` python
@asynccontextmanager
async def lifespan(app: FastAPI):
    global hub
    hub = init_hub()
    asyncio.create_task(run_pipeline_loop())
    yield
```

On startup: initialize memory hub, launch pipeline loop as background
task. The `yield` runs the server. Shutdown is handled by FastAPI after
yield.

**`/stream` endpoint (GET, SSE):**

Returns a `StreamingResponse` with `text/event-stream` content type. The
generator awaits events from `event_queue` (with 30-second timeout for
keepalive pings) and yields them as SSE-formatted data. The browser’s
`EventSource` API reconnects automatically if the connection drops.

**`/chat` endpoint (POST):**

Receives `{"message": str}`. Builds a rich context string from all
chains, recent alerts, and graph edges. Calls the Groq LLM with the
system prompt + context + operator question. Returns
`{"answer": str, "context_chains": int, "context_alerts": int}`.

**`/` endpoint (GET):**

Serves `ui/index.html` as an `HTMLResponse`. The UI has no build step —
it’s a single file with inline CSS and JS.

**Frame deduplication:**

``` python
already_processed = hub.sql.get_processed_frame_ids()
new_frames = [f for f in frames if f["frame_id"] not in already_processed]
```

The pipeline loop only processes frames not already in the database.
This means re-running the server on the same `frames.json` doesn’t
re-process all frames — only genuinely new ones are processed.

------------------------------------------------------------------------

#### `app/bus.py`

**Purpose:** Module-level asyncio.Queue pair.

``` python
observation_queue: asyncio.Queue = None
result_queue: asyncio.Queue = None

def init():
    global observation_queue, result_queue
    observation_queue = asyncio.Queue()
    result_queue      = asyncio.Queue()
```

Note: The bus queues are defined here but the actual event queue used in
`api.py` (`event_queue`) is declared locally in `api.py`. The `bus.py`
module was designed for the original worker-based architecture where
observations and results would flow through module-level queues. In the
current implementation, `process_frame()` uses `asyncio.gather()`
directly, so `bus.py` is defined but not actively used for the main
pipeline flow. It remains available for future extension.

------------------------------------------------------------------------

#### `ui/index.html`

**Purpose:** Single-file real-time dashboard. No framework, no build
step.

**Architecture:**

``` javascript
const es = new EventSource("/stream");
es.onmessage = (e) => {
    const payload = JSON.parse(e.data);
    if (payload.type === "done") { es.close(); return; }
    addCard(payload);  // Dynamically create DOM card
    updateStats();     // Update header counters
};
```

**Design features:** - Dark surveillance aesthetic (dark navy
background, muted blues/greens) - Cards animate in with CSS slide
animation on arrival - Alert cards have red/orange borders with
severity-based styling - Right-side chat panel sends `POST /chat` and
renders answers inline - Header shows live counts of total frames,
alerts, and active chains - Responsive to SSE `ping` events (keepalive —
no card rendered)

------------------------------------------------------------------------

#### `vlm/main.py`

**Purpose:** Standalone FastAPI server that processes uploaded video
files and generates `frames.json` using a real Vision Language Model.

**See Section 10 for full VLM deep dive.**

------------------------------------------------------------------------

## 6. Worker Concurrency & Coordination

### 6.1 Concurrent Worker Architecture

The system uses asynchronous concurrency to process each observation
through multiple independent pipelines simultaneously.

``` text
                    observation dict
                           │
                           ▼
                asyncio.gather(...)
        ┌────────────────┼────────────────┐
        ▼                ▼                ▼
retrieval_worker   synthesis_worker   chain_worker
        │                │                │
        ▼                ▼                ▼
vector retrieval   event synthesis   chain lookup
graph retrieval    event creation    dormant recovery
        │                │                │
        └────────────────┼────────────────┘
                         ▼
             context_assembler.resolve()
```

### 6.2 Worker Responsibilities

#### retrieval_worker

Purpose: retrieve historical semantic and graph context relevant to the
current observation.

**Operations performed:** - Retrieve semantically similar frames from
ChromaDB - Retrieve semantically similar events from ChromaDB - Retrieve
semantically similar chains from ChromaDB - Retrieve graph relationships
from NetworkX memory graph

**Output structure:**

``` python
{
    "similar_frames": [...],
    "similar_events": [...],
    "similar_chains": [...],
    "graph_relations": [...]
}
```

#### synthesis_worker

Purpose: convert the raw observation into a structured event
representation.

**Operations performed:** - Risk estimation - Event type
classification - Narrative synthesis - Event embedding creation

**Output structure:**

``` python
{
    "event_id": 11,
    "type": "suspicious_person_activity",
    "risk_score": 0.74,
    "description": "Repeated suspicious surveillance behavior",
    "embedding": [...]
}
```

#### chain_worker

Purpose: identify active or dormant chains that may relate to the
current observation.

**Operations performed:** - Query active chains from SQLite - Query
dormant chains within recovery window - Pre-filter incompatible chains -
Return candidate chains for scoring

**Output structure:**

``` python
{
    "active_chains": [...],
    "dormant_chains": [...]
}
```

### 6.3 Why Concurrency Matters

Without concurrency:

``` text
retrieval → synthesis → chain lookup
```

Each stage blocks the next.

With concurrency:

``` text
retrieval
     ┐
synthesis ├── run simultaneously
     ┘
chain lookup
```

Benefits: - Lower end-to-end latency - Vector retrieval overlaps with
synthesis computation - SQLite queries overlap with embedding
operations - Better CPU/GPU utilization - Faster UI updates

### 6.4 Synchronization Point

All workers synchronize at:

``` python
worker_results = await asyncio.gather(
    retrieval_worker.run(obs, hub),
    synthesis_worker.run(obs, hub),
    chain_worker.run(obs, hub)
)
```

The pipeline does not proceed until all workers complete.

Result ordering:

``` python
retrieval_results, event, chain_candidates = worker_results
```

### 6.5 Shared Resource Coordination

Workers share several resources safely:

| Shared Resource      | Access Pattern           |
|----------------------|--------------------------|
| SQLite connection    | serialized writes        |
| ChromaDB collections | concurrent reads/upserts |
| Embedding model      | concurrent inference     |
| NetworkX graph       | read-heavy, write-light  |
| Event hub            | shared orchestration     |

SQLite uses:

``` python
check_same_thread=False
```

to allow cross-thread async access.

### 6.6 Failure Isolation

Each worker failure is isolated independently.

Example:

``` python
try:
    similar_frames = frame_retriever.retrieve(obs)
except Exception:
    similar_frames = []
```

Effects: - Retrieval failure does not stop synthesis - Graph failure
does not stop chain resolution - Event synthesis failure can degrade
gracefully - Partial context is still usable

### 6.7 Event Hub Coordination

The `EventHub` object acts as the shared dependency container.

``` python
hub.sql
hub.vector
hub.graph
hub.embedder
hub.config
```

Workers receive the same hub reference:

``` python
retrieval_worker.run(obs, hub)
```

This avoids: - Re-opening DB connections - Re-loading embedding models -
Re-creating graph state - Duplicate ChromaDB clients

### 6.8 Backpressure Characteristics

Current behavior: - Frames processed sequentially - Workers concurrent
within each frame - No batching - No queue throttling

Effective execution pattern:

``` text
Frame 1:
    retrieval || synthesis || chain lookup

(wait complete)

Frame 2:
    retrieval || synthesis || chain lookup
```

This simplifies: - State consistency - Chain ordering - Event
chronology - Risk escalation logic

------------------------------------------------------------------------

## 7. Data Flow Analysis

### 7.1 Data Lifecycle Overview

``` text
Raw Input                  Processing                  Storage
─────────                  ──────────                  ───────
frames.json ──────────►  observation dict ──────────► SQLite: frames table
                              │                         ChromaDB: frames_collection
                              │
                              ▼
                         event dict ────────────────► SQLite: events table
                              │                         ChromaDB: events_collection
                              │
                              ▼
                         chain dict ────────────────► SQLite: chains table
                              │                         ChromaDB: chains_collection
                              │
                              ▼
                         alert dict ────────────────► SQLite: alerts table
                                                        NetworkX graph edges
```

### 7.2 Complete Frame Processing Flow

``` text
Frame N arrives
    │
    ├── [1] frame_input.load_frames() + telemetry_input.load_telemetry()
    │       Reads JSON files, filters to unprocessed frame IDs
    │
    ├── [2] observation_extractor.extract(frame, telemetry)
    │       ├── _classify(description, ACTIVITY_CATEGORIES)
    │       ├── _classify(description, ENTITY_CATEGORIES)
    │       ├── _extract_entity_fingerprint(description)
    │       ├── embed(fingerprint)
    │       ├── embed(description + location + activity)
    │       └── Returns obs dict with 14 fields
    │
    ├── [3] asyncio.gather(retrieval_worker, synthesis_worker, chain_worker)
    │       Running CONCURRENTLY:
    │       │
    │       ├── retrieval_worker.run(obs, hub)
    │       │   ├── frame_retriever.retrieve()
    │       │   ├── event_retriever.retrieve()
    │       │   ├── chain_retriever.retrieve()
    │       │   └── graph_retriever.retrieve()
    │       │
    │       ├── synthesis_worker.run(obs, hub)
    │       │   └── event_synthesizer.synthesize(obs, hub)
    │       │
    │       └── chain_worker.run(obs, hub)
    │           ├── get_active_and_dormant_chains()
    │           └── get_dormant_chains_within_window()
    │
    ├── [4] assemble_and_resolve_chain(obs, worker_results, hub)
    │       ├── Merge vector chains + SQL chains
    │       ├── Category compatibility gate
    │       ├── Risk floor gate
    │       ├── Security-response isolation gate
    │       ├── _score_chain_match()
    │       ├── Link/reactivate/create chain
    │       ├── Recalculate risk
    │       ├── Apply disposition risk cap
    │       └── Persist to all stores
    │
    ├── [5] risk_gate.should_reason(situation)
    │       ├── If risk < 0.45 → skip LLM
    │       └── Else → continue
    │
    ├── [6] llm_reasoner.reason(situation)
    │       ├── Build structured prompt
    │       ├── ChatGroq.invoke()
    │       ├── Parse JSON response
    │       └── Return reasoning output
    │
    ├── [7] alert_engine.process(situation, llm_output, hub)
    │       ├── update_event_llm_risk()
    │       ├── If risk < 0.70 → no alert
    │       └── If risk ≥ 0.70:
    │           ├── insert_alert()
    │           ├── update_chain(status='escalated')
    │           ├── graph.add_edges()
    │           └── graph.save()
    │
    └── [8] Push payload to event_queue → SSE → Browser UI
```

### 7.3 Observation Data Structure

``` python
{
    "frame_id": 5,
    "raw_description": "Unknown hooded figure standing near north perimeter fence",
    "activity": "standing and watching or surveilling",
    "activity_conf": 0.74,
    "entity_type": "person behaving suspiciously near perimeter",
    "entity_conf": 0.81,
    "entity_fingerprint": "Unknown hooded",
    "fingerprint_embedding": [0.12, -0.34, ...],
    "entities": ["person behaving suspiciously near perimeter"],
    "location": "North Perimeter",
    "time_str": "22:05",
    "time_minutes": 1325,
    "altitude": 9,
    "drone_id": "D-02",
    "embedding": [0.08, 0.23, ...]
}
```

### 7.4 Event Data Structure

``` python
{
    "event_id": 11,
    "frame_id": 9,
    "type": "suspicious_person_activity",
    "entities": ["person behaving suspiciously near perimeter"],
    "location": "North Perimeter",
    "risk_score": 0.74,
    "description": "Repeated suspicious nighttime surveillance behavior",
    "embedding": [0.11, -0.07, ...]
}
```

### 7.5 Situation / Chain Context Structure

``` python
{
    "observation": { ...14 fields... },
    "event": { ...event dict... },
    "chain": { ...chain dict... },
    "is_new_chain": False,
    "similar_frames": [
        {"id": "3", "metadata": {...}, "similarity": 0.82}
    ],
    "similar_events": [
        {"id": "7", "metadata": {...}, "similarity": 0.76}
    ],
    "graph_relations": [
        ("person behaving suspiciously near perimeter", "involved_in", "chain_2"),
        ("chain_2", "observed_at", "North Perimeter"),
    ],
}
```

### 7.6 Alert Data Structure

``` python
{
    "alert_id": 3,
    "chain_id": 2,
    "frame_id": 9,
    "event_id": 11,
    "risk_level": 0.88,
    "suspiciousness": "high",
    "reasoning": [
        "Entity matches dormant chain from prior session",
        "Repeated nighttime perimeter surveillance",
        "Graph shows prior escalation to alert"
    ],
    "recommended_action": "Dispatch security to North Perimeter immediately",
}
```

### 7.7 Producer / Consumer Matrix

| Data Artifact         | Producer                                         | Consumers                                   |
|------------------------|------------------------|------------------------|
| `frames.json`         | VLM/manual                                       | `frame_input.load_frames()`                 |
| `telemetry.json`      | Manual/simulated                                 | `telemetry_input.load_telemetry()`          |
| `observation dict`    | `observation_extractor.extract()`                | All workers, `context_assembler`            |
| `event dict`          | `event_synthesizer.synthesize()`                 | `context_assembler`, SQLite, ChromaDB       |
| `situation dict`      | `context_assembler.assemble_and_resolve_chain()` | `risk_gate`, `llm_reasoner`, `alert_engine` |
| `llm_output dict`     | `llm_reasoner.reason()`                          | `alert_engine.process()`                    |
| `alert dict`          | `alert_engine.process()`                         | SQLite, NetworkX graph, SSE UI              |
| `chain dict`          | `context_assembler`                              | SQLite, ChromaDB, NetworkX, chat context    |
| `embedding (384-dim)` | `embeddings.embed()`                             | ChromaDB upsert, chain scoring              |

### 7.8 Initialization State Logic

Every run checks:

``` python
already_processed = hub.sql.get_processed_frame_ids()
new_frames = [f for f in frames if f["frame_id"] not in already_processed]
```

Behavior:

-   **First run:** all frames processed
-   **Second run with same inputs:** nothing reprocessed
-   **Extended frames.json:** only new frames processed
-   **Dormant chains:** previous chains become reactivation candidates

## 8. End-to-End Execution Flow

### 8.1 CLI Mode Execution Flow

    python -m app.pipeline
              │
              ▼
    1. init_hub()
       ├── makedirs data/db, data/chroma, data/graph
       ├── MemoryHub(DB_PATH, CHROMA_PATH, GRAPH_PATH)
       │   ├── SQLiteStore(DB_PATH) → CREATE TABLE IF NOT EXISTS (×4 tables)
       │   ├── VectorStore(CHROMA_PATH) → ChromaDB client, get_or_create 3 collections
       │   └── GraphStore(GRAPH_PATH) → load pickle if exists, else empty DiGraph
       └── mark_old_chains_dormant(3600)
           └── UPDATE chains SET status='dormant' WHERE active AND last_updated < (now - 1h)

    2. load_frames("data/simulation/frames.json")
       └── Returns 30 frame dicts

    3. load_telemetry("data/simulation/telemetry.json")
       └── Returns {frame_id: telemetry_dict} for 30 frames

    4. For each frame (30 iterations):
       │
       ├── extract(frame, telemetry)
       │   ├── _classify(description, ACTIVITY_CATEGORIES) → activity, conf, _
       │   ├── _classify(description, ENTITY_CATEGORIES)   → entity_type, conf, _
       │   ├── _extract_entity_fingerprint(description)    → fingerprint str
       │   ├── embed(fingerprint) → fingerprint_embedding (384-dim)
       │   └── embed(f"{description} {location} {activity}") → embedding (384-dim)
       │       Returns: observation dict (14 fields)
       │
       └── process_frame(observation, hub)
           │
           ├── asyncio.create_task(retrieval_worker.run(...))   ──┐
           ├── asyncio.create_task(synthesis_worker.run(...))   ──┤ concurrent
           ├── asyncio.create_task(chain_worker.run(...))       ──┘
           │
           ├── asyncio.gather(r_task, s_task, c_task)
           │   │
           │   ├── retrieval_worker.run():
           │   │   ├── frame_retriever.retrieve()
           │   │   │   └── hub.vector.query_frames(obs.embedding, top_k=5)
           │   │   │       → filter(similarity >= 0.55) → similar_frames
           │   │   ├── event_retriever.retrieve()
           │   │   │   └── hub.vector.query_events(obs.embedding, top_k=3) → similar_events
           │   │   ├── chain_retriever.retrieve()
           │   │   │   └── hub.vector.query_chains(obs.embedding, top_k=3, floor=0.45)
           │   │   │       → candidate_chains_vector
           │   │   └── graph_retriever.retrieve()
           │   │       └── hub.graph.get_relations_for_entities(obs.entities)
           │   │           → graph_relations (list of triples)
           │   │
           │   ├── synthesis_worker.run():
           │   │   └── synthesize(observation, hub)
           │   │       ├── _semantic_risk_score(obs.embedding, observation) → base_risk
           │   │       ├── contextual bonuses (night, restricted, known entity)
           │   │       ├── security response suppressor (clear → 0.25)
           │   │       └── embed(event_type + location + description) → event.embedding
           │   │           Returns: event dict
           │   │
           │   └── chain_worker.run():
           │       ├── hub.sql.get_active_and_dormant_chains() → recent_chains_sql
           │       └── hub.sql.get_dormant_chains_within_window(86400) → dormant_candidates_sql
           │
           ├── assemble_and_resolve_chain(observation, worker_results, hub)
           │   ├── Merge vector_chains + sql_chains by chain_id
           │   ├── For each candidate chain:
           │   │   ├── GATE: category mismatch? skip
           │   │   ├── GATE: event risk ≤ 0.35? skip
           │   │   ├── GATE: security response + suspicious chain? skip
           │   │   └── _score_chain_match() → combined score (fp+entity+location+vector)
           │   ├── best_chain = highest scoring candidate
           │   │
           │   ├── IF best_match >= 0.60:
           │   │   ├── IF dormant → reactivate (status=active, fetch past events)
           │   │   ├── Update event_ids, risk, locations, entities
           │   │   ├── _update_disposition(chain, event)
           │   │   └── _recalculate_risk() then _apply_disposition_risk_cap()
           │   │
           │   └── ELSE: create new chain
           │       ├── narrative = activity + location
           │       ├── _initial_disposition(obs, event)
           │       └── _apply_disposition_risk_cap()
           │
           ├── Persist to all 3 stores:
           │   ├── hub.sql.insert_event(event) → event_id
           │   ├── hub.sql.insert_chain OR update_chain → chain_id
           │   ├── hub.sql.update_event_chain(event_id, chain_id)
           │   ├── hub.sql.insert_frame(observation)
           │   ├── hub.vector.upsert_chain(chain_id, embedding, metadata)
           │   ├── hub.vector.upsert_event(event_id, embedding, metadata)
           │   ├── hub.vector.upsert_frame(frame_id, embedding, metadata)
           │   ├── hub.graph.add_edges(triples)
           │   └── hub.graph.save() → memory.gpickle
           │
           ├── should_reason(situation) → True/False
           │   └── chain.risk_score >= 0.45?
           │
           ├── IF True: reason(situation) → llm_output
           │   ├── Format prompt (observation + chain context + historical)
           │   ├── ChatGroq.invoke() → raw JSON string
           │   └── _parse_response(raw) → dict
           │
           └── process_alert(situation, llm_output, hub)
               ├── update_event_llm_risk(event_id, risk_level)
               ├── IF risk_level < 0.70: return None
               └── IF risk_level >= 0.70:
                   ├── insert_alert(alert_dict) → alert_id
                   ├── update_chain(status=escalated)
                   ├── graph.add_edges(chain→alert, alert→event)
                   └── graph.save()
                   Returns: alert dict

    5. Print result to console
       ├── Alert: "🚨 ALERT — Frame N | Risk: X | Level: HIGH | Action: ..."
       └── Normal: "Frame N — activity @ location — risk=X — chain=Y (status)"

### 8.2 Web (API) Mode Execution Flow

    uvicorn app.api:app --reload --port 8000
              │
              ▼
    FastAPI lifespan startup:
    ├── init_hub() (same as CLI)
    └── asyncio.create_task(run_pipeline_loop())
        [pipeline runs as background task]

    Browser connects to http://localhost:8000/
    ├── GET / → serves ui/index.html
    └── Browser JS: new EventSource("/stream")
        └── GET /stream → StreamingResponse (SSE)
            └── Awaits events from event_queue

    run_pipeline_loop() (background):
    ├── load frames and telemetry
    ├── Filter to new_frames only (not already in DB)
    ├── For each new frame:
    │   ├── extract → process_frame (same as CLI)
    │   ├── Build payload dict
    │   ├── await event_queue.put(payload)  ← SSE receives this
    │   └── await asyncio.sleep(1.5)  ← simulated real-time delay
    └── await event_queue.put({"type": "done"})

    Browser receives SSE event:
    ├── Parse JSON payload
    ├── addCard(payload) → create DOM card with animation
    ├── If alert: red border, alert badge, reasoning shown
    └── updateStats() → header counters

    Operator sends chat message:
    ├── POST /chat {"message": "Are there any threats?"}
    ├── _build_surveillance_context() → rich plaintext context
    │   ├── Query all chains (sorted by risk)
    │   ├── Fetch frame descriptions per chain
    │   ├── Fetch recent alerts
    │   └── Extract graph triples
    ├── Groq LLM call (context + question → answer)
    └── Return {"answer": str, "context_chains": N, "context_alerts": M}

### 8.3 Lifecycle State Diagram

              NEW RUN                     SAME INPUTS AGAIN
                 │                              │
                 ▼                              ▼
        init_hub()                      init_hub()
                 │                              │
        mark_old_chains_dormant(3600)   mark_old_chains_dormant(3600)
                 │                              │
        [no chains exist]               [prior chains → dormant]
                 │                              │
        Frame 5 (hooded man, 22:05):    Frame 5 (same description):
        → No candidates                 → Matches dormant chain from prior run
        → New chain created (id=1)      → Chain reactivated (status=active)
        → status=neutral, risk=0.56     → Risk escalates from prior level
                 │                              │
        Frame 7 (hooded man, 22:28):    Frame 7:
        → Matches chain 1 (fp sim)      → Continues reactivated chain
        → Links, risk escalates to 0.72 → Risk escalates further
                 │                              │
        should_reason → True            should_reason → True
        Groq call → risk 0.85           Groq call → higher risk (more context)
                 │                              │
        ALERT generated                 ALERT generated (reactivation context)

------------------------------------------------------------------------

## 9. Full Data Flow Analysis

### 9.1 Data Origin to Persistence

    frames.json                     telemetry.json
        │                               │
        └──────────────┬────────────────┘
                       │
                  observation_extractor.extract()
                       │
                       ▼
             observation dict (14 fields)
             {frame_id, raw_description, activity,
              entity_type, entities, embedding,
              fingerprint, fingerprint_embedding, ...}
                       │
             ┌─────────┼─────────────────────────┐
             │         │                         │
             ▼         ▼                         ▼
      retrieval     synthesis              chain worker
      worker        worker                      │
         │              │                       │
         │              ▼                       ▼
         │         event dict            sql_chains list
         │         {type, entities,
         │          risk_score,
         │          embedding}
         │              │
         └──────────────┴────────────────────┐
                                             │
                                     assemble_and_resolve_chain()
                                             │
                                             ▼
                                  resolved_chain dict
                                  {chain_id, status, narrative,
                                   entities, locations, event_ids,
                                   risk_score, disposition,
                                   embedding, fingerprint_embedding}
                                             │
                        ┌────────────────────┼──────────────────────┐
                        │                    │                       │
                        ▼                    ▼                       ▼
                 SQLite store          ChromaDB store         NetworkX graph
                 insert_frame()        upsert_frame()         add_edges()
                 insert_event()        upsert_event()         save()
                 insert/update_chain() upsert_chain()
                 insert_alert()

### 9.2 Key Data Structures

**Observation dict** (created by `observation_extractor.extract()`):

``` python
{
    "frame_id": 5,
    "raw_description": "Unknown hooded figure standing near north perimeter fence",
    "activity": "standing and watching or surveilling",
    "activity_conf": 0.74,
    "entity_type": "person behaving suspiciously near perimeter",
    "entity_conf": 0.81,
    "entity_fingerprint": "Unknown hooded",
    "fingerprint_embedding": [0.12, -0.34, ...],  # 384-dim
    "entities": ["person behaving suspiciously near perimeter"],
    "location": "North Perimeter",
    "time_str": "22:05",
    "time_minutes": 1325,
    "altitude": 9,
    "drone_id": "D-02",
    "embedding": [0.08, 0.23, ...],  # 384-dim
}
```

**Event dict** (created by `event_synthesizer.synthesize()`):

``` python
{
    "event_id": None,           # assigned by SQLite on insert
    "frame_id": 5,
    "chain_id": None,           # assigned by assembler
    "type": "standing_and_watching_or_surveilling",
    "entities": ["person behaving suspiciously near perimeter"],
    "location": "North Perimeter",
    "time_str": "22:05",
    "time_minutes": 1325,
    "risk_score": 0.59,         # base semantic + bonuses
    "llm_risk": None,           # filled by alert_engine if LLM runs
    "embedding": [0.15, 0.41, ...],  # 384-dim event embedding
}
```

**Chain dict** (managed by `context_assembler.py`):

``` python
{
    "chain_id": 2,
    "status": "suspicious",         # active / dormant / escalated / suspicious
    "disposition": "suspicious",    # benign / neutral / suspicious
    "narrative": "standing and watching or surveilling at North Perimeter",
    "entities": ["person behaving suspiciously near perimeter"],
    "locations": ["North Perimeter"],
    "event_ids": [5, 7, 9, 23, 25], # frame_ids in this chain
    "risk_score": 0.83,
    "entity_fingerprint": "Unknown hooded",
    "fingerprint_embedding": [...],
    "embedding": [...],             # chain narrative embedding
    "_past_events": [...],          # transient: loaded on reactivation
    "created_at": "2026-05-16 22:05:00",
    "last_updated": "2026-05-16 22:25:00",
}
```

**Situation dict** (assembled by
`context_assembler.assemble_and_resolve_chain()`):

``` python
{
    "observation": { ...14 fields... },
    "event": { ...event dict... },
    "chain": { ...chain dict... },
    "is_new_chain": False,
    "similar_frames": [{"id": "3", "metadata": {...}, "similarity": 0.82}],
    "similar_events": [{"id": "7", "metadata": {...}, "similarity": 0.76}],
    "graph_relations": [
        ("person behaving suspiciously near perimeter", "involved_in", "chain_2"),
        ("chain_2", "observed_at", "North Perimeter"),
    ],
}
```

**Alert dict** (created by `alert_engine.process()`):

``` python
{
    "alert_id": 3,
    "chain_id": 2,
    "frame_id": 9,
    "event_id": 11,
    "risk_level": 0.88,
    "suspiciousness": "high",
    "reasoning": [
        "Entity matches dormant chain from prior session",
        "Repeated nighttime perimeter surveillance",
        "Graph shows prior escalation to alert"
    ],
    "recommended_action": "Dispatch security to North Perimeter immediately",
}
```

### 9.3 Producer/Consumer Map

| Data Artifact         | Producer                                         | Consumers                                            |
|------------------------|------------------------|------------------------|
| `frames.json`         | VLM/manual                                       | `frame_input.load_frames()`                          |
| `telemetry.json`      | Manual/simulated                                 | `telemetry_input.load_telemetry()`                   |
| `observation dict`    | `observation_extractor.extract()`                | All 3 workers, `context_assembler`                   |
| `event dict`          | `event_synthesizer.synthesize()`                 | `context_assembler`, SQLite, ChromaDB                |
| `situation dict`      | `context_assembler.assemble_and_resolve_chain()` | `risk_gate`, `llm_reasoner`, `alert_engine`          |
| `llm_output dict`     | `llm_reasoner.reason()`                          | `alert_engine.process()`                             |
| `alert dict`          | `alert_engine.process()`                         | SQLite, NetworkX graph, `api.py` (SSE), UI           |
| `chain dict`          | `context_assembler`                              | SQLite, ChromaDB, NetworkX, LLM prompt, chat context |
| `embedding (384-dim)` | `embeddings.embed()`                             | ChromaDB upsert, chain scoring                       |

------------------------------------------------------------------------

## 10. Storage & State Management

### 10.1 SQLite — Ground Truth

**File:** `data/db/surveillance.db`  
**Connection:** Persistent single connection,
`check_same_thread=False`  
**Purpose:** Authoritative record of everything that has happened

**Write lifecycle:** - `insert_frame()`: called once per frame,
immediately after extraction - `insert_event()`: called after synthesis,
before chain resolution - `update_event_chain()`: called after chain
resolution (backfills chain_id) - `update_event_llm_risk()`: called by
alert engine after LLM run - `insert_chain()` or `update_chain()`:
called after chain decision - `insert_alert()`: called if risk \>= 0.70

**Read lifecycle:** - `get_active_and_dormant_chains()`: called by
chain_worker on every frame - `get_dormant_chains_within_window()`:
called by chain_worker on every frame - `get_events_for_chain()`: called
on chain reactivation (loads full event history) -
`get_processed_frame_ids()`: called once at API startup
(deduplication) - `get_recent_alerts()`: called by `/chat` endpoint for
context building

**Cleanup/retention:** No automatic cleanup. The database grows
indefinitely. In a production system, you’d add archival/rotation for
frames older than N days.

**Rerun behavior:** `INSERT OR REPLACE INTO frames` means re-inserting a
frame is idempotent. The API’s frame deduplication prevents
re-processing entirely. The CLI would re-process and re-insert,
potentially creating duplicate events for the same chain.

### 10.2 ChromaDB — Semantic Similarity Index

**Path:** `data/chroma/`  
**Client:** `PersistentClient` — writes to disk automatically  
**Purpose:** Fast approximate nearest-neighbor search by embedding

**Three collections and their roles:**

`frames_collection`: Every frame’s observation embedding (activity +
location + description). Used by `frame_retriever` to answer “What past
frames were semantically similar to what the drone sees now?” Historical
context for the LLM prompt.

`events_collection`: Every event’s semantic embedding (event_type +
location + description). Used by `event_retriever` to answer “What past
behavioral events resemble the current observation?” More abstract than
frames — event-level patterns.

`chains_collection`: Every chain’s narrative embedding (narrative +
entities + locations). Used by `chain_retriever` to answer “Are there
dormant or active chains whose overall behavioral story matches this
observation?” The primary signal for chain linking.

**Upsert semantics:** `collection.upsert()` is idempotent by ID.
Re-inserting a frame/event/chain replaces the prior entry. This is
correct for chains (which evolve) and for reactivation (which refreshes
the chain embedding).

**Query format returned:**

``` python
{"id": "14", "metadata": {"type": "...", "location": "...", ...}, "similarity": 0.82}
```

**ChromaDB version note:** Version 0.5.3 (pinned in requirements) has a
telemetry bug. The fix is applied at the top of `vector_store.py`.

### 10.3 NetworkX Graph — Relationship Memory

**File:** `data/graph/memory.gpickle`  
**Representation:** `nx.DiGraph()` (directed graph)  
**Lifecycle:** Loaded once at startup, lives in memory, saved after
every write

**Node types:** - Entity strings:
`"person behaving suspiciously near perimeter"`,
`"vehicle parked or stationary"` - Event strings: `"event_5"`,
`"event_12"` - Chain strings: `"chain_2"`, `"chain_4"` - Alert strings:
`"alert_1"`, `"alert_3"` - Location strings: `"North Perimeter"`,
`"Garage Sector"`

**When the graph answers questions SQLite can’t:**

SQLite can answer “How many events are in chain 2?” easily. But “Is this
entity linked to any chain that previously escalated to an alert?”
requires joining frames → events → chains → alerts with transitive
logic. NetworkX answers this with:

``` python
nx.ego_graph(G, entity, radius=2)
# Returns: all nodes within 2 hops — linked chains, past alerts, related events
```

**graph_retriever query in practice:**

For entity `"person behaving suspiciously near perimeter"` that has been
seen before, the ego-graph at radius=2 returns:

    person → involved_in → chain_2
    chain_2 → escalated_to → alert_1
    chain_2 → observed_at → North Perimeter
    event_5 → part_of → chain_2

This context goes directly into the LLM prompt, allowing the LLM to
reason: “This entity was previously linked to a chain that escalated to
an alert.”

### 10.4 Embedding Cache

**Location:** In-process dictionaries in `observation_extractor.py` and
`context_assembler.py`

The category embeddings (ACTIVITY_CATEGORIES, ENTITY_CATEGORIES for
extractor; HIGH/LOW risk descriptors for synthesizer) are computed once
on first use and cached:

``` python
_category_embeddings: dict = {}
def _get_category_embeddings(categories: list) -> list:
    key = tuple(categories)
    if key not in _category_embeddings:
        _category_embeddings[key] = [embed(c) for c in categories]
    return _category_embeddings[key]
```

This prevents re-embedding the same fixed category strings on every
frame. With 12 activity categories and 9 entity categories, this saves
21 embed() calls per frame — significant given each embed() call runs
neural inference.

------------------------------------------------------------------------

## 11. API, Service & Pipeline Breakdown

### 11.1 FastAPI Endpoints

#### `GET /`

**Purpose:** Serves the dashboard UI  
**Handler:** Opens `ui/index.html` and returns as `HTMLResponse`  
**No authentication, no parameters**

------------------------------------------------------------------------

#### `GET /stream`

**Purpose:** Real-time event feed via Server-Sent Events  
**Content-Type:** `text/event-stream`  
**Headers:** `Cache-Control: no-cache`, `X-Accel-Buffering: no`

**Event types emitted:**

| `type` field | When                   | Contents                                                              |
|------------------------|------------------------|------------------------|
| `"frame"`    | Normal frame processed | description, location, chain_id, risk_score, activity, suspiciousness |
| `"alert"`    | Alert generated        | All frame fields + alert dict with reasoning/action                   |
| `"done"`     | Pipeline loop complete | Signal to stop listening                                              |
| `"ping"`     | 30-second timeout      | Keepalive, browser ignores                                            |

**Request lifecycle:** 1. Browser opens `EventSource("/stream")` on page
load 2. FastAPI streams from `event_queue` (async generator) 3. Each
`await event_queue.get()` yields when the pipeline puts a new result 4.
`asyncio.wait_for(..., timeout=30)` prevents stalled connections 5. On
`"done"` event, generator returns and stream closes

------------------------------------------------------------------------

#### `POST /chat`

**Purpose:** Operator query interface — answers natural language
questions about the surveillance state  
**Request body:** `{"message": str}`  
**Response body:**
`{"answer": str, "context_chains": int, "context_alerts": int}`

**Context building (`_build_surveillance_context()`):**

For each chain (up to 15, sorted by risk_score descending): - Chain ID,
status, risk score - Narrative description - Entity types and
locations - Event count - Last 3 raw frame descriptions (so operator can
ask “what about the hooded man?” and the LLM matches it to frame
descriptions)

For each recent alert (up to 10): - Alert ID, chain ID, risk level,
suspiciousness - First reasoning point - Recommended action

For graph edges (up to 20): - Entity relationship triples

This rich context is injected into the system prompt, enabling the LLM
to answer questions like: - “Is there currently a threat?” (checks risk
scores and escalated chains) - “What happened near the garage?” (matches
location) - “Did the blue truck come back?” (matches frame description
text) - “What was alert 3 about?” (directly references alert data)

------------------------------------------------------------------------

### 11.2 VLM Service

**Endpoint:** `POST /upload-video` (running at separate port, default
FastAPI port if launched separately)  
**Request:** Multipart file upload (MP4)  
**Response:**
`{"status": "done", "json_file": "outputs/frames.json", "frames_processed": N}`

**See Section 10 for full VLM details.**

------------------------------------------------------------------------

### 11.3 LangChain Chain (Prompt → LLM → Parser)

The LangChain usage in this project is intentionally minimal — only
`PromptTemplate | ChatGroq | StrOutputParser` is used. No LangChain
memory, no LangChain agents, no LangChain tools.

``` python
llm    = ChatGroq(model=GROQ_MODEL, api_key=GROQ_API_KEY, max_tokens=512, temperature=0.2)
prompt = PromptTemplate.from_template(PROMPT_TEMPLATE)
chain_ = prompt | llm | StrOutputParser()
result = chain_.invoke(input_vars)
```

The `|` syntax is LangChain’s LCEL (LangChain Expression Language) for
composing runnables. Each step transforms the output of the previous
step.

------------------------------------------------------------------------

## 12. Component Interaction Matrix

| Component               | Receives From                           | Sends To                                      | Data Shared                                                                  | Purpose                             |
|---------------|---------------|---------------|---------------|---------------|
| `frame_input`           | `frames.json`                           | `pipeline.run()`                              | `list[frame_dict]`                                                           | Load raw frames                     |
| `telemetry_input`       | `telemetry.json`                        | `pipeline.run()`                              | `dict[frame_id, telem]`                                                      | Load telemetry                      |
| `observation_extractor` | `frame_dict`, `telem_dict`              | All 3 workers                                 | `observation dict (14 fields)`                                               | Semantic classification + embedding |
| `embeddings.embed()`    | Text string                             | All callers                                   | 384-dim float list                                                           | Single embedding source             |
| `retrieval_worker`      | `observation`, `memory_hub`             | `context_assembler`                           | `{similar_frames, similar_events, candidate_chains_vector, graph_relations}` | Historical context fetch            |
| `synthesis_worker`      | `observation`, `memory_hub`             | `context_assembler`                           | `{event dict}`                                                               | Risk scoring + event creation       |
| `chain_worker`          | `observation`, `memory_hub`             | `context_assembler`                           | `{recent_chains_sql, dormant_candidates_sql}`                                | SQL chain state snapshot            |
| `context_assembler`     | All 3 worker results                    | `risk_gate`, storage                          | `situation dict`                                                             | Chain resolution + persistence      |
| `memory_hub`            | `context_assembler`, all workers        | All storage                                   | Unified storage access                                                       | Dependency injection                |
| `sqlite_store`          | `memory_hub`                            | `context_assembler`, `chain_worker`, `api.py` | 4 tables                                                                     | Authoritative record                |
| `vector_store`          | `memory_hub`                            | All retrievers, `context_assembler`           | 3 ChromaDB collections                                                       | Semantic search                     |
| `graph_store`           | `memory_hub`                            | `graph_retriever`, `context_assembler`        | NetworkX DiGraph                                                             | Relationship traversal              |
| `risk_gate`             | `situation dict`                        | `llm_reasoner` (conditional)                  | `bool`                                                                       | LLM call decision                   |
| `llm_reasoner`          | `situation dict`                        | `alert_engine`                                | `{suspiciousness, risk_level, reasoning, recommended_action, confidence}`    | Threat assessment                   |
| `alert_engine`          | `situation`, `llm_output`, `memory_hub` | SQLite, NetworkX, `event_queue`               | `alert dict or None`                                                         | Alert generation                    |
| `api.py`                | `event_queue`, POST `/chat`             | Browser (SSE), chat response                  | `payload dict`, `answer str`                                                 | UI backend                          |
| `bus.py`                | (defined, reserved)                     | (unused in current flow)                      | asyncio.Queue pair                                                           | Future event bus                    |
| `vlm/main.py`           | Uploaded video file                     | `frames.json`                                 | Frame descriptions                                                           | Real video processing               |

------------------------------------------------------------------------

## 13. VLM Pipeline & frames.json Deep Dive

### 13.1 The Two-Mode Design

The system has two modes for producing `frames.json`:

**Mode 1 — Simulation (default):** A pre-written
`data/simulation/frames.json` with 30 hand-crafted frame descriptions
representing realistic security scenarios. Used for development, demos,
and testing without video hardware.

**Mode 2 — Real VLM (optional):** Upload an actual surveillance video to
the VLM service at `vlm/main.py`. The VLM extracts frames, runs them
through a vision-language model, and generates a `frames.json` with
AI-captioned descriptions.

### 13.2 VLM Architecture (`vlm/main.py`)

    Video File Upload (MP4)
             │
             ▼
        /upload-video
             │
             ▼
        save to uploads/UUID.mp4
             │
             ▼
        process_video(video_path)
             │
             ├── cv2.VideoCapture(video_path)
             │
             └── LOOP: frame_count % 30 == 0
                 │
                 ├── frame_rgb = cv2.cvtColor(frame, BGR→RGB)
                 ├── image = Image.fromarray(frame_rgb)
                 ├── image.resize(320, 240)  ← smaller for faster inference
                 │
                 ├── inputs = processor(
                 │       text="You are a surveillance AI system. Describe...",
                 │       images=image,
                 │       return_tensors="pt"
                 │   )
                 │
                 ├── output = model.generate(**inputs, max_new_tokens=40)
                 │
                 └── result = processor.batch_decode(output, skip_special_tokens=True)[0]
                     → Appended to results list with frame_id
             │
             ▼
        json.dump(results, outputs/frames.json)
             │
             ▼
        Return {"status": "done", "frames_processed": N}

### 13.3 VLM Model Choice

**Model:** `HuggingFaceTB/SmolVLM-256M-Instruct`

This is a 256-million parameter vision-language model — extremely small
by modern standards. It was chosen because: - Downloads in seconds, not
hours - Runs on CPU without GPU - Produces concise surveillance-style
descriptions adequate for the pipeline - No API costs (fully local
inference)

At 256M parameters, it’s less capable than larger models (GPT-4V, Claude
3) but for a surveillance demo context, “Unknown person walking near
fence” is sufficient description for the downstream semantic
classification to work correctly.

**Why sample every 30th frame?**

A typical surveillance video at 30 FPS produces 30 frames per second.
Processing every frame would be: - Redundant (adjacent frames are nearly
identical) - Extremely slow on CPU (each inference takes seconds)

Sampling every 30th frame gives 1 description per second of video —
enough temporal resolution for surveillance without computational
infeasibility.

**Image resize to 320×240:**

The original surveillance frame might be 1920×1080. Running VLM
inference on full resolution is memory-intensive and slow. 320×240
preserves enough detail for the model to identify persons, vehicles, and
activity while dramatically reducing inference time.

### 13.4 `frames.json` Structure

``` json
[
    {"frame_id": 1, "description": "Empty parking lot, no activity detected"},
    {"frame_id": 2, "description": "Security guard walking east wing perimeter"},
    {"frame_id": 5, "description": "Unknown hooded figure standing near north perimeter fence"},
    ...
]
```

**Key design decisions:**

-   `frame_id` is sequential starting at 1 (not a video timestamp — the
    pipeline uses `telemetry.json` for time)
-   `description` is free-form text — no structured fields, no object
    coordinates
-   The main pipeline doesn’t care whether descriptions came from VLM or
    from the simulation file — the observation extractor processes them
    identically

**Why text descriptions, not image tensors?**

The main pipeline uses fastembed (text-only) for embeddings. Feeding
image tensors directly into the pipeline would require a multimodal
embedding model. The design choice was to keep the VLM as a
preprocessing step that converts visual information to text, which then
feeds the text-native pipeline. This is the “VLM-inspired” approach
mentioned in the original assignment — clean separation between visual
understanding and behavioral analysis.

### 13.5 Telemetry Correlation

`frames.json` and `telemetry.json` are separate files keyed by
`frame_id`. The VLM generates `frames.json`. The operator must
separately provide or generate `telemetry.json` with corresponding frame
IDs and drone metadata.

In simulation mode, both files are pre-written with matching IDs. In
real video mode, the operator would need to either: - Record drone
telemetry alongside the video (standard drone flight log) - Estimate
telemetry from video metadata (GPS tags, timestamp)

### 13.6 Downstream Consumption

After `frames.json` is generated (VLM or simulation), it flows:

    frames.json → frame_input.load_frames() → list[{frame_id, description}]
    telemetry.json → telemetry_input.load_telemetry() → dict[frame_id, telemetry]

    For each frame:
        frame_dict + telemetry_dict → observation_extractor.extract()
        → observation dict → pipeline processing

The observation extractor doesn’t know or care whether the description
came from a 256M VLM or was hand-written. It classifies semantically
regardless.

------------------------------------------------------------------------

## 14. Async Operations, Loops & Event Systems

### 14.1 Core Async Architecture

The system uses Python’s `asyncio` single-threaded event loop
throughout. There are no threads, no multiprocessing. All concurrency is
cooperative — tasks yield to each other on `await` points.

### 14.2 Per-Frame Parallel Workers

``` python
async def process_frame(observation: dict, memory_hub) -> dict:
    r_task = asyncio.create_task(retrieval_worker.run(copy.deepcopy(observation), memory_hub))
    s_task = asyncio.create_task(synthesis_worker.run(copy.deepcopy(observation), memory_hub))
    c_task = asyncio.create_task(chain_worker.run(copy.deepcopy(observation), memory_hub))

    results = await asyncio.gather(r_task, s_task, c_task)
```

**`copy.deepcopy(observation)`:** Each worker receives an independent
copy of the observation dict. If workers modified the observation dict
in place (unlikely but possible), one worker’s modification wouldn’t
affect another’s input. This is defensive programming for correctness in
the concurrent context.

**`asyncio.gather()`:** Fires all three tasks and awaits all of them.
Returns when all three complete. The overall latency is
`max(retrieval_time, synthesis_time, chain_time)` rather than their sum.

**Why are the workers async if they don’t do IO?**

The workers are declared `async def run(...)` even though they don’t
`await` anything (ChromaDB queries and SQLite queries use synchronous
drivers). This is intentional: making them async tasks allows
`asyncio.gather()` to schedule them as coroutines. In practice, they run
one at a time (cooperative multitasking) rather than in true parallel.
True parallelism would require `loop.run_in_executor()` with a
ThreadPoolExecutor. For a prototype, the sequential cooperative
execution is acceptable.

### 14.3 Pipeline Loop (API Mode)

``` python
async def run_pipeline_loop():
    for frame in new_frames:
        obs    = extract(frame, telem)
        result = await process_frame(obs, hub)
        await event_queue.put(payload)
        await asyncio.sleep(1.5)   # ← simulates real-time drone feed
    await event_queue.put({"type": "done"})
```

`asyncio.sleep(1.5)` yields control to the event loop, allowing the SSE
generator to drain the queue and send events to the browser. Without
this sleep, all frames would be processed instantly, the queue would
fill up, and the browser would receive all events in a burst rather than
a stream.

**In production (real drone):** Replace the sleep with actual drone
polling/streaming logic. The 1.5s delay is for demo aesthetics only.

### 14.4 SSE Generator Loop

``` python
async def event_generator():
    while True:
        try:
            event = await asyncio.wait_for(event_queue.get(), timeout=30)
            yield f"data: {json.dumps(event)}\n\n"
            if event.get("type") == "done":
                break
        except asyncio.TimeoutError:
            yield 'data: {"type":"ping"}\n\n'
```

**`asyncio.wait_for(queue.get(), timeout=30)`:** If no event arrives
within 30 seconds, a ping is sent. This prevents the HTTP connection
from timing out on the client side. Browsers typically close SSE
connections after 60 seconds of silence.

**Why yield `\n\n` after data?** The SSE protocol requires
`data: <content>\n\n` (two newlines) to terminate each event. A single
`\n` would be a line continuation within an event.

### 14.5 Chat Endpoint (Concurrent with Pipeline)

``` python
@app.post("/chat")
async def chat(req: ChatRequest):
    context = _build_surveillance_context()
    # ...Groq LLM call...
    return {"answer": answer, ...}
```

The chat endpoint is a completely independent async handler. When a user
sends a chat message: 1. FastAPI receives the POST request 2. A new
async task is spawned for this handler 3. The pipeline loop is still
running in its own task 4. Both tasks are scheduled cooperatively — the
chat call yields during the Groq API await, allowing pipeline steps to
proceed 5. Chat response and pipeline processing are effectively
concurrent (interleaved at await points)

**No locking is needed** because SQLite reads are concurrent (multiple
readers are fine) and the graph and vector stores are read-only during
the chat call.

------------------------------------------------------------------------

## 15. Error Handling, Edge Cases & Pitfalls

### 15.1 LLM Fallback

**Problem:** Groq API is unavailable, rate-limited, or returns malformed
JSON.

**Implementation:**

``` python
try:
    raw = chain_.invoke(input_vars)
    result = _parse_response(raw)
    return result
except Exception as e:
    logger.error(f"LLM reasoning failed: {e} — using fallback")
    return {
        "suspiciousness": "medium",
        "risk_level": chain.get("risk_score", 0.5),
        "reasoning": ["LLM unavailable — rule-based risk used"],
        "recommended_action": "Review manually",
        "confidence": "low",
    }
```

**Why:** The system must continue processing even if the LLM is
unavailable. The fallback uses the rule-based chain risk score, which is
less nuanced but still functional. The confidence field signals to
operators that the assessment is degraded.

**Parsing defense:**

``` python
def _parse_response(text: str) -> dict:
    text = text.strip()
    if text.startswith("```"):
        lines = text.split("\n")
        text = "\n".join(lines[1:-1]) if len(lines) > 2 else text
    return json.loads(text)
```

Some LLM calls return JSON wrapped in markdown code fences
(```` ```json\n...\n``` ````). This strips them before parsing.

### 15.2 ChromaDB Telemetry Bug

**Problem:** ChromaDB 0.5.3 emits
`capture() takes 1 positional argument but 3 were given` on every query.
This is a version compatibility issue in ChromaDB’s internal telemetry
module.

**Fix:**

``` python
import os, logging
os.environ["ANONYMIZED_TELEMETRY"] = "false"
logging.getLogger("chromadb.telemetry").setLevel(logging.ERROR)
logging.getLogger("chromadb").setLevel(logging.WARNING)
```

Both environment variable and logger suppression are applied as
belt-and-suspenders.

### 15.3 Empty ChromaDB Collection Queries

**Problem:** Querying an empty ChromaDB collection raises an exception
rather than returning empty results.

**Fix:**

``` python
def _query(self, collection, embedding: list, top_k: int) -> list:
    try:
        count = collection.count()
        if count == 0:
            return []
        k = min(top_k, count)
        results = collection.query(query_embeddings=[embedding], n_results=k)
        ...
    except Exception as e:
        logger.error(f"ChromaDB query failed: {e}")
        return []
```

Guards both the empty collection case (count == 0 → return early) and
any other query failure (catch-all exception → return empty list rather
than crash).

### 15.4 Chain Linking Failures (Historical Issue)

**Problem:** Initially, entity matching used substring overlap. “Hooded
man” and “hooded figure” were treated as different entities.

**Fix:** Semantic classification against fixed categories. All variants
of a hooded person map to
`"person behaving suspiciously near perimeter"`. The chain match scorer
then uses embedding similarity between these category strings, not
substring matching.

**Lesson:** Brittle string matching is the wrong approach for free-text
surveillance descriptions. Semantic similarity is the correct tool.

### 15.5 Risk De-escalation on Repeat Presence (Historical Issue)

**Problem:** A vehicle’s chain risk dropped when the vehicle appeared
again with lower individual event risk.
`new_risk = 0.7 * chain_risk + 0.3 * event_risk` means a chain at 0.495
drops when linked to a 0.20-risk event.

**Fix:** `repeat_pressure` bonus (`0.07 * event_count`), de-escalation
floor at 97% of prior risk, and aggressive escalation weighting for
suspicious chains.

**Lesson:** Chain risk should monotonically increase (or stay flat) on
repeated suspicious presence. De-escalation should be slow and
deliberate, not automatic.

### 15.6 Security Guard Chains

**Problem:** Security guard patrol observations were being classified as
low-risk threats and occasionally building up chain risk.

**Fix:** Three-layer suppression: 1. `event_synthesizer.py` forces
base_risk to 0.25 for security/patrol observations 2.
`context_assembler.py` prevents security response observations from
linking to suspicious chains 3. `disposition = "benign"` with risk cap
0.32 ensures the chain never crosses LLM threshold

### 15.7 Frame Deduplication on Re-run

**Problem:** Re-running the API server on unchanged `frames.json` would
re-process all 30 frames, creating duplicate events.

**Fix:**

``` python
already_processed = hub.sql.get_processed_frame_ids()
new_frames = [f for f in frames if f["frame_id"] not in already_processed]
```

Frames already in the database are skipped. The pipeline only processes
genuinely new frames. This is idiomatic for a continuous drone feed
where frame IDs always increment.

### 15.8 Dormant Chain State on Restart

**Problem:** Prior-session chains remained `active` on restart. New
observations would link to them incorrectly (active linking
vs. reactivation logic are different code paths).

**Fix:** `mark_old_chains_dormant(3600)` at startup. Any chain not
updated in the last hour becomes dormant. This triggers the full
reactivation logic (fetch past events, log “Reactivating chain N”) on
the first matching new observation.

------------------------------------------------------------------------

## 16. Engineering Tradeoffs & Design Decisions

### 16.1 Why fastembed over sentence-transformers?

**Chosen:** fastembed (`BAAI/bge-small-en`, \~50MB)  
**Rejected:** sentence-transformers (requires torch, \~2GB download)

**Tradeoff:** sentence-transformers offers larger and more capable
models. fastembed trades model quality for install size and speed. For
surveillance text descriptions (short, domain-specific), the quality
difference is negligible. Install time drops from 10+ minutes to under 2
minutes.

### 16.2 Why ChromaDB over FAISS?

**Chosen:** ChromaDB (persistent, metadata-aware)  
**Rejected:** FAISS (fast, file-based, requires custom persistence)

**Tradeoff:** FAISS is faster at query time for very large vector sets
(millions). ChromaDB has slightly higher per-query overhead but provides
persistence and metadata filtering (e.g., “only search chains with
status=dormant”) out of the box. For a prototype with hundreds to
thousands of vectors, ChromaDB’s convenience outweighs FAISS’s speed
advantage.

### 16.3 Why Direct Groq SDK (via LangChain adapter) over full LangChain?

**Chosen:** `langchain-groq` (thin adapter),
`PromptTemplate | ChatGroq | StrOutputParser`  
**Rejected:** Full LangChain stack (LangChain agents, LangChain memory,
LangChain tools)

**Tradeoff:** LangChain adds \~500ms import time, introduces three
abstraction layers, and would require LangChain-specific patterns for
memory (which would conflict with the custom three-store memory system).
Using only the LCEL chain builder keeps the benefits (prompt
composition, output parsing) without the cost.

### 16.4 Why Three Storage Systems?

Each storage system answers a fundamentally different class of query:

-   **SQLite:** “Give me all events in chain 3, ordered by time.” Exact,
    structured, relational.
-   **ChromaDB:** “What past observations are semantically similar to
    this one?” Approximate, vector-based.
-   **NetworkX:** “Is this entity linked to any chain that escalated to
    an alert?” Traversal, multi-hop.

No single storage system handles all three efficiently. SQLite can’t do
semantic search. ChromaDB can’t do exact relational queries. NetworkX
can’t do similarity search. Using all three is the correct architectural
choice, not overengineering.

### 16.5 Why Semantic Classification Over Keyword Matching?

**Keyword matching** breaks on paraphrase. “Hooded man”, “masked
individual”, “suspicious figure in dark clothing” are all the same
concept but different strings.

**Semantic classification** embeds the input and compares against
embedded category descriptions. Paraphrases cluster in embedding space,
making them robust to wording variation. The category list defines the
vocabulary — new variants of known concepts are correctly classified
without code changes.

The tradeoff: category lists must be thoughtfully designed. A category
missing from the list can’t be classified (falls to the closest
available category). The current lists cover the primary surveillance
scenarios adequately.

### 16.6 Why Disposition System (benign/neutral/suspicious)?

Without dispositions, security guards and delivery trucks accumulate
event counts and risk scores just like suspicious entities. A security
guard doing 5 patrol loops would approach the LLM threshold purely from
event count pressure.

Dispositions create a hard semantic separation: authorized entities are
`benign` by creation, capped at 0.32 (below the 0.45 LLM threshold), and
never generate alerts. Suspicious entities are `suspicious` with no cap.
Unknown entities are `neutral` — they can become suspicious if evidence
accumulates.

The tradeoff: the system must correctly classify entity types on first
observation. If a suspicious person is misclassified as authorized
(unlikely given the semantic categories but possible), they’d be marked
benign and never trigger an alert.

### 16.7 Why SSE over WebSocket?

**Chosen:** Server-Sent Events (SSE)  
**Rejected:** WebSocket

**Tradeoff:** WebSocket is bidirectional and more powerful. SSE is
unidirectional (server → client) and simpler. The dashboard only needs
server → client (pipeline events). The chat is handled by a separate
POST endpoint. SSE is sufficient, has native browser support via
`EventSource`, and requires no client-side WebSocket library.

------------------------------------------------------------------------

## 17. Dependency & Relationship Mapping

### 17.1 Module Import Dependencies

    config.py (no imports from app/)
        ↓ imported by: observation_extractor, event_synthesizer,
                       context_assembler, risk_gate, llm_reasoner,
                       alert_engine, pipeline, api

    app/utils/embeddings.py
        ↓ imported by: observation_extractor, event_synthesizer,
                       context_assembler, all retriever_base consumers

    app/utils/logger.py
        ↓ imported by: every module in app/

    app/memory/sqlite_store.py }
    app/memory/vector_store.py } → app/memory/memory_hub.py
    app/memory/graph_store.py  }
        ↓ imported by: context_assembler (via hub), chain_worker (via hub),
                       alert_engine (via hub), api (via hub)

    app/retrieval/retriever_base.py
        ↓ imported by: frame_retriever, event_retriever, chain_retriever

    app/retrieval/{frame,event,chain,graph}_retriever.py
        ↓ imported by: retrieval_worker

    app/workers/{retrieval,synthesis,chain}_worker.py
        ↓ imported by: pipeline

    app/engine/event_synthesizer.py → synthesis_worker
    app/engine/context_assembler.py → pipeline
    app/engine/risk_gate.py         → pipeline

    app/reasoning/llm_reasoner.py → pipeline
    app/alerts/alert_engine.py    → pipeline

    app/pipeline.py (process_frame, init_hub) → api.py
    app/ingestion/*.py → pipeline, api

### 17.2 Data Ownership

| Data              | Owner            | Others can read          | Others can write                     |
|------------------|------------------|------------------|------------------|
| `frames.json`     | Manual/VLM       | `frame_input`            | No                                   |
| `telemetry.json`  | Manual/simulated | `telemetry_input`        | No                                   |
| `surveillance.db` | `sqlite_store`   | `chain_worker`, `api.py` | `context_assembler` via sqlite_store |
| `data/chroma/`    | `vector_store`   | All retrievers           | `context_assembler` via vector_store |
| `memory.gpickle`  | `graph_store`    | `graph_retriever`        | `context_assembler` via graph_store  |
| `event_queue`     | `api.py`         | —                        | `run_pipeline_loop()`                |

### 17.3 Call Chain Hierarchy

    pipeline.run() / api.run_pipeline_loop()
        └── process_frame(observation, hub)
            ├── retrieval_worker.run()
            │   ├── frame_retriever.retrieve()
            │   │   └── memory_hub.vector.query_frames()
            │   ├── event_retriever.retrieve()
            │   │   └── memory_hub.vector.query_events()
            │   ├── chain_retriever.retrieve()
            │   │   └── memory_hub.vector.query_chains()
            │   └── graph_retriever.retrieve()
            │       └── memory_hub.graph.get_relations_for_entities()
            ├── synthesis_worker.run()
            │   └── event_synthesizer.synthesize()
            │       ├── _semantic_risk_score()
            │       └── embeddings.embed()
            ├── chain_worker.run()
            │   ├── memory_hub.sql.get_active_and_dormant_chains()
            │   └── memory_hub.sql.get_dormant_chains_within_window()
            ├── context_assembler.assemble_and_resolve_chain()
            │   ├── _merge_chain_candidates()
            │   ├── _score_chain_match() for each candidate
            │   ├── _recalculate_risk()
            │   ├── _update_disposition()
            │   ├── _apply_disposition_risk_cap()
            │   ├── memory_hub.sql.insert_event()
            │   ├── memory_hub.sql.insert/update_chain()
            │   ├── memory_hub.vector.upsert_*() × 3
            │   ├── memory_hub.graph.add_edges()
            │   └── memory_hub.graph.save()
            ├── risk_gate.should_reason()
            ├── llm_reasoner.reason() [conditional]
            │   └── ChatGroq.invoke()
            └── alert_engine.process()
                ├── memory_hub.sql.update_event_llm_risk()
                ├── memory_hub.sql.insert_alert()
                ├── memory_hub.sql.update_chain()
                ├── memory_hub.graph.add_edges()
                └── memory_hub.graph.save()

------------------------------------------------------------------------

## 18. Developer Onboarding Guide

### 18.1 Prerequisites

-   Python 3.11+ (asyncio features, `dict | None` type union syntax)
-   A free Groq API key from
    [console.groq.com](https://console.groq.com)
-   \~1GB disk space for ChromaDB and fastembed model cache

### 18.2 Installation

``` bash
# 1. Clone or unzip the project
cd srvlnc_agent_clean

# 2. Create virtual environment (recommended)
python -m venv venv
source venv/bin/activate   # Linux/Mac
venv\Scripts\activate      # Windows

# 3. Install dependencies (takes 2-3 minutes)
pip install -r requirements.txt

# 4. Create .env file
echo "GROQ_API_KEY=your_key_here" > .env
```

### 18.3 Running the System

**CLI mode (batch processing, terminal output):**

``` bash
python -m app.pipeline
```

**Web mode (dashboard + chat):**

``` bash
uvicorn app.api:app --reload --port 8000
# Open http://localhost:8000 in browser
```

**VLM service (optional, for real video processing):**

``` bash
# From the vlm/ directory
cd vlm
uvicorn main:app --port 8001
# Then POST a video to http://localhost:8001/upload-video
```

### 18.4 Fresh Start vs. Persistent Mode

**Clean slate (wipe all memory):**

``` bash
rm -f data/db/surveillance.db data/graph/memory.gpickle
rm -rf data/chroma/
python -m app.pipeline
```

**Continue from prior run (persistent memory, chains reactivate):**

``` bash
python -m app.pipeline
# Prior chains become dormant, reactivate on matching frames
```

**Add new frames (expanding frame IDs):**

``` bash
# Add new frames to data/simulation/frames.json with new IDs (31, 32, ...)
# Add matching telemetry to data/simulation/telemetry.json
python -m app.pipeline
# Only new frame IDs are processed (API mode deduplicates)
```

### 18.5 Configuration Tuning

All tunable parameters are in `config.py`. Common adjustments:

``` python
LLM_RISK_THRESHOLD  = 0.45  # Lower → more LLM calls (more sensitive)
ALERT_THRESHOLD     = 0.70  # Lower → more alerts generated
CHAIN_LINK_THRESHOLD = 0.60  # Lower → more aggressive chain linking
CHAIN_REACTIVATION_WINDOW = 86400  # Seconds — increase for longer memory
_BENIGN_RISK_CAP    = 0.32  # In context_assembler.py — ceiling for authorized actors
```

Add new restricted locations:

``` python
RESTRICTED_LOCATIONS = ["North Perimeter", "Garage Sector", "Your New Area"]
```

Add adjacency relationships:

``` python
LOCATION_ADJACENCY = {
    "Your New Area": ["Adjacent Zone 1", "Adjacent Zone 2"],
}
```

### 18.6 Adding New Frames

Edit `data/simulation/frames.json`:

``` json
{"frame_id": 31, "description": "Your frame description here"}
```

Edit `data/simulation/telemetry.json`:

``` json
{"frame_id": 31, "time": "HH:MM", "location": "Zone Name", "altitude": 10, "drone_id": "D-01"}
```

Frame descriptions can use any natural language. The semantic classifier
handles novel descriptions.

### 18.7 Adding New Semantic Categories

To add a new activity category, edit `ACTIVITY_CATEGORIES` in
`app/ingestion/observation_extractor.py`:

``` python
ACTIVITY_CATEGORIES = [
    ...,
    "person using electronic device near secure area",  # new category
]
```

The category embeddings are cached on first use — the next pipeline run
will incorporate the new category automatically.

### 18.8 Debugging Workflow

**See classification decisions:**

``` bash
# Set log level to DEBUG in logger.py or watch console output
# Each frame logs: [activity] entity=[entity_type] fingerprint=[...] @ location
```

**Inspect chain state:**

``` bash
sqlite3 data/db/surveillance.db "SELECT chain_id, status, narrative, risk_score, disposition FROM chains ORDER BY risk_score DESC;"
```

**Inspect recent alerts:**

``` bash
sqlite3 data/db/surveillance.db "SELECT * FROM alerts;"
```

**Check graph size:**

``` python
import pickle, networkx as nx
G = pickle.load(open("data/graph/memory.gpickle", "rb"))
print(G.number_of_nodes(), G.number_of_edges())
list(G.edges(data=True))[:10]
```

**Check ChromaDB collection counts:**

``` python
import chromadb
client = chromadb.PersistentClient(path="data/chroma/")
for coll_name in ["frames_collection", "events_collection", "chains_collection"]:
    print(coll_name, client.get_collection(coll_name).count())
```

### 18.9 Safe Modification Areas

**Safe to modify without side effects:** - `config.py` — all threshold
and path changes - `data/simulation/frames.json` and `telemetry.json` —
add/edit frame data - `ACTIVITY_CATEGORIES` and `ENTITY_CATEGORIES` in
`observation_extractor.py` — adding categories - `HIGH_RISK_DESCRIPTORS`
and `LOW_RISK_DESCRIPTORS` in `event_synthesizer.py` — tuning risk

**Modify with care:** - `context_assembler.py` — chain scoring weights
affect linking behavior system-wide - `_BENIGN_RISK_CAP` and disposition
thresholds — affect which chains get alerts - `llm_reasoner.py`
PROMPT_TEMPLATE — changes LLM output format (update parser accordingly)

**Test after modifying:** - Delete all storage and re-run with full
30-frame simulation - Verify: benign frames (1, 2, 6) stay low risk -
Verify: hooded person chain (frames 5, 7, 9, 23, 25) escalates to
alert - Verify: frame 28 (security response) creates its own chain, not
linked to suspicious chain

------------------------------------------------------------------------

## 19. Future Improvements

### 19.1 Architecture Improvements

**True parallel workers with thread pool:**

``` python
import asyncio, concurrent.futures
pool = concurrent.futures.ThreadPoolExecutor(max_workers=4)

async def process_frame(observation, hub):
    loop = asyncio.get_event_loop()
    r = loop.run_in_executor(pool, retrieval_worker.run_sync, observation, hub)
    s = loop.run_in_executor(pool, synthesis_worker.run_sync, observation, hub)
    c = loop.run_in_executor(pool, chain_worker.run_sync, observation, hub)
    results = await asyncio.gather(r, s, c)
```

This would give true parallelism for the workers (currently
cooperative).

**Persistent event stream for reconnection:** Currently, if a browser
disconnects and reconnects, it misses events emitted during
disconnection. Storing events in a ring buffer (or Redis Streams) would
allow reconnecting clients to catch up.

**Multi-drone support:** The current pipeline processes frames
sequentially. Multiple drones would require either: (a) a queue per
drone with separate pipeline loops, or (b) frame batching with
drone_id-aware chain isolation (a chain for “North Perimeter Drone D-02”
shouldn’t merge with “Garage Drone D-01”).

### 19.2 Scalability

**Replace SQLite with PostgreSQL:** For multi-user, high-throughput
deployments. SQLite’s single-writer limitation becomes a bottleneck at
scale.

**Replace NetworkX pickle with a graph database:** Neo4j or ArangoDB for
production-scale graph storage with concurrent read access and proper
transaction support.

**Replace ChromaDB with Qdrant or Weaviate:** For higher-volume vector
search with better filtering, index management, and distributed
deployment.

**Async SQLite:** Use `aiosqlite` for true non-blocking SQLite access.
The current synchronous SQLite in an async context works but ties up the
event loop during writes.

### 19.3 Observability

**Structured logging with correlation IDs:** Add a `frame_id`
correlation ID to every log line so a frame’s entire processing flow can
be traced in a log aggregator.

**Metrics endpoint (`/metrics`):** Expose Prometheus-compatible metrics:
frames processed/minute, alert rate, LLM call rate, average risk score
per chain, ChromaDB query latency.

**Audit trail:** Every chain state transition (active → dormant →
reactivated → escalated) should be logged to a separate `chain_events`
table with timestamps. This provides a full audit trail of system
decisions.

### 19.4 Security

**API authentication:** The current `/chat` and `/stream` endpoints have
no authentication. In production, add API key authentication or JWT
tokens for the operator interface.

**Input validation:** Frame descriptions from VLM should be sanitized
before processing — a maliciously crafted description could potentially
manipulate risk scores via carefully designed text.

**Rate limiting:** The `/chat` endpoint calls the Groq API for every
request. Add rate limiting to prevent operator chat from exhausting API
quotas.

### 19.5 VLM Improvements

**Larger VLM model:** Replace SmolVLM-256M with a 2-7B parameter model
for significantly better frame descriptions. Consider Qwen-VL, LLaVA, or
Phi-3-Vision.

**Frame sampling strategy:** Instead of every 30th frame, use motion
detection (OpenCV frame differencing) to only sample frames with
significant activity. This dramatically reduces processing for long
inactive periods.

**Batch inference:** Process multiple frames in a single model forward
pass for GPU efficiency.

**Direct pipeline integration:** Instead of generating `frames.json` as
an intermediate file, stream frame descriptions directly into the
observation queue as they’re generated.

### 19.6 Maintainability

**Type hints throughout:** Add full Python type annotations to all
function signatures for IDE support and static analysis.

**Test coverage:** Add integration tests that: - Verify benign frames
don’t generate alerts - Verify the hooded person chain escalates
correctly over 5 frames - Verify dormant chain reactivation works as
expected - Verify security response isolation

**Configuration validation:** Add Pydantic model for `config.py` to
validate types and ranges on startup, with helpful error messages if
configuration is invalid.

------------------------------------------------------------------------

## 20. Glossary

| Term                            | Definition                                                                                                                                                             |
|------------------------------------|------------------------------------|
| **Behavioral Chain**            | A sequence of related security events linked by entity, location, and time. A hooded figure observed across multiple frames forms one chain.                           |
| **Dormant Chain**               | A chain that was active but has had no new events for more than `CHAIN_REACTIVATION_WINDOW` seconds. Eligible for reactivation if a similar observation appears.       |
| **Reactivation**                | When a new observation matches a dormant chain, the chain is made active again and accumulates new events, building on prior context.                                  |
| **Disposition**                 | A chain attribute (benign/neutral/suspicious) that controls risk caps and LLM call eligibility.                                                                        |
| **Risk Gate**                   | The function `should_reason()` that decides whether to call the LLM based on a chain’s risk score vs `LLM_RISK_THRESHOLD`.                                             |
| **Entity Fingerprint**          | A short specific phrase extracted from a frame description (“Hooded man”, “Blue pickup truck”) used for fine-grained identity matching within broad entity categories. |
| **Semantic Classification**     | Classifying input text by embedding similarity against a fixed set of natural language category descriptions, rather than keyword matching.                            |
| **SSE (Server-Sent Events)**    | A browser API (`EventSource`) for receiving a unidirectional stream of events from the server over HTTP. Used for the real-time dashboard.                             |
| **situation dict**              | The central data structure passed between pipeline stages, containing the observation, synthesized event, resolved chain, historical context, and graph relations.     |
| **VLM (Vision-Language Model)** | A multimodal model that processes images and generates text descriptions. Used in `vlm/main.py` to caption actual surveillance video frames.                           |
| **repeat_pressure**             | A small risk escalation bonus applied per chain event count, ensuring that repeated suspicious presence steadily increases risk rather than averaging it down.         |
| **fastembed**                   | A lightweight embedding library by Qdrant that loads small embedding models without requiring PyTorch or the full `transformers` library.                              |
| **ChromaDB**                    | An embedded vector database that persists to disk and supports metadata filtering alongside embedding similarity search.                                               |
| **BAAI/bge-small-en**           | The specific embedding model used — 384-dimensional dense vector encoder, \~50MB, optimized for English text similarity.                                               |
| **ego_graph**                   | A NetworkX function that returns all nodes within N hops of a given node in a graph. Used for multi-hop relationship queries.                                          |
| **memory_hub**                  | A single Python object that holds references to all three storage systems (SQLite, ChromaDB, NetworkX). Passed as dependency injection to all components.              |
| **LangChain LCEL**              | LangChain Expression Language — a `|` pipeline syntax for composing prompt, LLM, and parser into a single runnable chain.                                              |

------------------------------------------------------------------------

